In [4]:
# Keep repository-relative paths valid from notebook subfolders.
from pathlib import Path
import os

os.chdir(next(
    root for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "notebooks").is_dir() and (root / "requirements.txt").is_file()
))

# CELL 1 - IMPORT AND INSPECT DATA DIRECTORY

from pathlib import Path

import pandas as pd
import numpy as np


DATA_DIR = Path("data")

print(
    "Data directory exists:",
    DATA_DIR.exists()
)

print("\nFILES IN DATA DIRECTORY")

for path in sorted(
    DATA_DIR.iterdir()
):
    if path.is_file():

        size_mb = (
            path.stat().st_size
            / 1024
            / 1024
        )

        print(
            f"{path.name:<60} "
            f"{size_mb:>10.2f} MB"
        )

Data directory exists: True

FILES IN DATA DIRECTORY
idx_dataset_inventory.csv                                         17.91 MB
idx_financial_benchmark_by_period.csv                              0.01 MB
idx_financial_benchmark_overall.csv                                0.00 MB
idx_financial_currency_scale_checkpoint.csv                        2.28 MB
idx_financial_currency_scale_mapping.csv                           2.29 MB
idx_financial_currency_scale_mapping_final.csv                     2.29 MB
idx_financial_current_metric_conflicts.csv                         0.33 MB
idx_financial_current_metrics_clean.csv                           22.34 MB
idx_financial_current_metrics_final.csv                           23.66 MB
idx_financial_current_metrics_selected.csv                        21.71 MB
idx_financial_file_scan.csv                                        7.55 MB
idx_financial_label_discovery.csv                                176.70 MB
idx_financial_label_discovery_checkpoint.csv   

In [5]:
# CELL 2 - FIND LARGE DATA FILES

supported_extensions = {
    ".csv",
    ".parquet",
    ".json",
    ".jsonl"
}


large_files = []


for path in DATA_DIR.rglob("*"):

    if (
        path.is_file()
        and path.suffix.lower()
        in supported_extensions
    ):

        size_bytes = (
            path.stat().st_size
        )

        size_mb = (
            size_bytes
            / 1024
            / 1024
        )

        large_files.append(
            {
                "file": str(path),
                "extension":
                    path.suffix.lower(),
                "size_mb":
                    size_mb
            }
        )


large_files_df = (
    pd.DataFrame(
        large_files
    )
)


if len(large_files_df) > 0:

    large_files_df = (
        large_files_df
        .sort_values(
            "size_mb",
            ascending=False
        )
        .reset_index(drop=True)
    )


display(
    large_files_df.head(30)
)

,file,extension,size_mb
0,data\company_dataset\company_dataset_raw.csv,.csv,1590.532363
1,data\idx_financial_label_discovery.csv,.csv,176.700492
2,data\idx_financial_label_discovery_checkpoint.csv,.csv,176.700492
3,data\idx_financial_period_mapped_corrected.csv,.csv,52.613593
4,data\idx_financial_period_mapping_checkpoint.csv,.csv,44.997947
5,data\idx_financial_metric_candidates.csv,.csv,41.672852
6,data\idx_financial_metric_candidates_enriched.csv,.csv,30.312409
7,data\idx_financial_metrics_normalized_final.csv,.csv,27.385677
8,data\idx_financial_current_metrics_final.csv,.csv,23.660724
9,data\idx_financial_current_metrics_clean.csv,.csv,22.341969


In [6]:
# CELL 3 - SET LARGE DATASET FILE

LARGE_DATASET_FILE = Path(
    "data/company_dataset/raw/company_dataset_raw.csv"
)

print(
    "Dataset file:",
    LARGE_DATASET_FILE
)

print(
    "Exists:",
    LARGE_DATASET_FILE.exists()
)

if LARGE_DATASET_FILE.exists():

    print(
        "Size MB:",
        round(
            LARGE_DATASET_FILE.stat().st_size
            / 1024
            / 1024,
            2
        )
    )

    print(
        "Extension:",
        LARGE_DATASET_FILE.suffix.lower()
    )

Dataset file: data\company_dataset\company_dataset_raw.csv
Exists: True
Size MB: 1590.53
Extension: .csv


In [7]:
# CELL 4 - READ SMALL SAMPLE AND INSPECT SCHEMA

file_extension = (
    LARGE_DATASET_FILE
    .suffix
    .lower()
)


if file_extension == ".csv":

    sample_df = pd.read_csv(
        LARGE_DATASET_FILE,
        nrows=100,
        low_memory=False
    )


elif file_extension == ".parquet":

    sample_df = pd.read_parquet(
        LARGE_DATASET_FILE
    ).head(100)


elif file_extension in {
    ".jsonl",
    ".json"
}:

    try:

        sample_df = pd.read_json(
            LARGE_DATASET_FILE,
            lines=True,
            nrows=100
        )

    except Exception:

        sample_df = (
            pd.read_json(
                LARGE_DATASET_FILE
            )
            .head(100)
        )


else:

    raise ValueError(
        f"Unsupported file type: "
        f"{file_extension}"
    )


print(
    "Sample rows:",
    len(sample_df)
)

print(
    "Columns:",
    len(sample_df.columns)
)


print("\nCOLUMN NAMES")

for column in sample_df.columns:
    print(column)


print("\nSAMPLE DTYPES")

print(
    sample_df.dtypes
)


display(
    sample_df.head(10)
)

Sample rows: 100
Columns: 41

COLUMN NAMES
id
created_at
name
short_description
semrush_global_rank
semrush_visits_latest_month
num_investors
funding_total
num_exits
num_funding_rounds
last_funding_type
last_funding_at
num_acquisitions
apptopia_total_apps
apptopia_total_downloads
contact_email
phone_number
facebook
linkedin
twitter
num_investments
num_lead_investments
num_lead_investors
listed_stock_symbol
company_type
hub_tags
operating_status
founded_on
categories
founders
website
ipo_status
num_employees_enum
locations
growth_insight_description
growth_insight_indicator
growth_insight_direction
growth_insight_confidence
investor_insight_description
permalink
url

SAMPLE DTYPES
id                                  str
created_at                          str
name                                str
short_description                   str
semrush_global_rank             float64
semrush_visits_latest_month     float64
num_investors                   float64
funding_total                  

,id,created_at,name,short_description,semrush_global_rank,semrush_visits_latest_month,num_investors,funding_total,num_exits,num_funding_rounds,...,ipo_status,num_employees_enum,locations,growth_insight_description,growth_insight_indicator,growth_insight_direction,growth_insight_confidence,investor_insight_description,permalink,url
0,a8de17a2-5700-40d0-8331-9f8fa7b3df27,2024-08-18 09:47:54.407151+00,GivingTech,"GivingTech - Giving Technologies, Inc is creat...",1801734.0,12368.0,NaN,NaN,NaN,NaN,...,private,c_00001_00010,"Europe, Middle East, and Africa (EMEA), Middle...",NaN,NaN,NaN,NaN,NaN,𝗚𝗶𝘃𝗶𝗻𝗴𝗧𝗲𝗰h,https://www.crunchbase.com/organization/𝗚𝗶𝘃𝗶𝗻𝗴...
1,dfa41d55-72f7-4745-8ba9-a1fe664e90f8,2024-08-24 18:14:46.291141+00,X+,X+ is an elite club for DeGods whales and buil...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,𝐗,https://www.crunchbase.com/organization/𝐗
2,5d7e4faf-7880-47eb-bd72-902057733280,2024-08-18 06:49:06.951712+00,F. Rego,"F. Rego offers insurance consulting, brokerage...",5228054.0,1140.0,NaN,NaN,NaN,NaN,...,private,c_00051_00100,"European Union (EU), Europe, Middle East, and ...",NaN,NaN,NaN,NaN,NaN,𝗙-𝗥𝗲𝗴𝗼,https://www.crunchbase.com/organization/𝗙-𝗥𝗲𝗴𝗼
3,075b9784-a35f-4682-9422-412e6791bde0,2024-08-20 10:20:19.463579+00,O’dara Exotic Skincare,O’dara Exotic Skincare is a skincare line that...,NaN,NaN,1.0,0.0,NaN,1.0,...,private,NaN,"Greater Los Angeles Area, West Coast, Western US",NaN,NaN,NaN,NaN,NaN,𝐎-𝐝𝐚𝐫𝐚-𝐄𝐱𝐨𝐭𝐢𝐜-𝐒𝐤𝐢𝐧𝐜𝐚𝐫e,https://www.crunchbase.com/organization/𝐎-𝐝𝐚𝐫𝐚...
4,ee90241c-d768-4167-a428-feefda59da90,2024-08-24 17:07:32.901299+00,!Bewust sociaal op web,!Bewust sociaal op web is providing social med...,NaN,NaN,NaN,NaN,NaN,NaN,...,private,c_00001_00010,"European Union (EU), Europe, Middle East, and ...",NaN,NaN,NaN,NaN,NaN,bewust-sociaal-op-web,https://www.crunchbase.com/organization/bewust...
5,9a0d9dd1-8024-41f7-984b-45ac32997339,2024-08-17 09:52:46.624066+00,!Creatice,!Creatice is a Education based company.,NaN,NaN,NaN,302000.0,NaN,1.0,...,private,c_00011_00050,Asia-Pacific (APAC),NaN,NaN,NaN,NaN,NaN,creatice,https://www.crunchbase.com/organization/creatice
6,bce54cde-ca3b-41be-81b9-c9a1d01e4e8c,2024-08-17 15:57:48.866428+00,!DOEVE*,"!DOEVE* offers marketing, communication and sa...",NaN,NaN,NaN,NaN,NaN,NaN,...,private,c_00001_00010,"European Union (EU), Europe, Middle East, and ...",NaN,NaN,NaN,NaN,NaN,doeve,https://www.crunchbase.com/organization/doeve
7,d4b0e8f9-86f2-022d-6d9a-598f1a7033cf,2024-08-18 03:37:00.299812+00,!FEST,Fest is a company that creates a unique atmosp...,1117555.0,26885.0,NaN,NaN,NaN,NaN,...,private,c_00101_00250,"Europe, Middle East, and Africa (EMEA)",NaN,NaN,NaN,NaN,NaN,fest,https://www.crunchbase.com/organization/fest
8,8e2c2fbd-2244-32eb-8e35-9ea9cefeb90f,2024-08-19 07:52:08.916809+00,!K7,A music video production company based in Berlin.,2104309.0,9147.0,NaN,NaN,NaN,NaN,...,private,c_00011_00050,"European Union (EU), Europe, Middle East, and ...",NaN,NaN,NaN,NaN,NaN,k7-2,https://www.crunchbase.com/organization/k7-2
9,7268c6ad-9c37-3265-7545-0f4c07fe22a2,2024-08-22 14:38:10.198084+00,!SHOUTTAG,A !shouttag is a powerful call to action that ...,NaN,NaN,NaN,NaN,NaN,NaN,...,private,c_00001_00010,"Greater Los Angeles Area, Inland Empire, West ...",NaN,NaN,NaN,NaN,NaN,shouttag,https://www.crunchbase.com/organization/shouttag


In [8]:
# CELL 5 - COUNT ROWS AND AUDIT BASIC DATA QUALITY USING CHUNKS

CHUNK_SIZE = 100_000


total_rows = 0

missing_counts = {
    column: 0
    for column in sample_df.columns
}

non_missing_counts = {
    column: 0
    for column in sample_df.columns
}


for chunk_number, chunk in enumerate(
    pd.read_csv(
        LARGE_DATASET_FILE,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    total_rows += len(chunk)

    for column in chunk.columns:

        missing = (
            chunk[column]
            .isna()
            .sum()
        )

        missing_counts[column] += missing

        non_missing_counts[column] += (
            len(chunk) - missing
        )


    if chunk_number % 5 == 0:

        print(
            f"Processed chunks: {chunk_number} "
            f"| rows: {total_rows:,}"
        )


print("\nTOTAL ROWS")

print(
    f"{total_rows:,}"
)


column_quality_df = pd.DataFrame(
    {
        "column":
            list(missing_counts.keys()),

        "missing_count":
            list(missing_counts.values()),

        "non_missing_count":
            list(non_missing_counts.values())
    }
)


column_quality_df[
    "missing_pct"
] = (
    column_quality_df[
        "missing_count"
    ]
    /
    total_rows
    *
    100
)


column_quality_df = (
    column_quality_df
    .sort_values(
        "missing_pct",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    column_quality_df
)

Processed chunks: 5 | rows: 500,000
Processed chunks: 10 | rows: 1,000,000
Processed chunks: 15 | rows: 1,500,000
Processed chunks: 20 | rows: 2,000,000
Processed chunks: 25 | rows: 2,500,000

TOTAL ROWS
2,807,492


,column,missing_count,non_missing_count,missing_pct
0,num_exits,2802295,5197,99.814888
1,hub_tags,2800803,6689,99.761745
2,num_lead_investments,2800709,6783,99.758396
3,listed_stock_symbol,2794934,12558,99.552697
4,num_investments,2794545,12947,99.538841
5,growth_insight_confidence,2773720,33772,98.797076
6,growth_insight_direction,2773720,33772,98.797076
7,investor_insight_description,2773174,34318,98.777628
8,num_acquisitions,2759214,48278,98.280387
9,apptopia_total_downloads,2754306,53186,98.105569


In [9]:
# CELL 6 - CHECK DUPLICATE COMPANY IDS

seen_ids = set()

duplicate_id_count = 0
missing_id_count = 0

duplicate_id_samples = []


for chunk_number, chunk in enumerate(
    pd.read_csv(
        LARGE_DATASET_FILE,
        usecols=[
            "id"
        ],
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    ids = (
        chunk[
            "id"
        ]
        .dropna()
        .astype(str)
    )


    missing_id_count += (
        chunk[
            "id"
        ]
        .isna()
        .sum()
    )


    for company_id in ids:

        if company_id in seen_ids:

            duplicate_id_count += 1

            if len(
                duplicate_id_samples
            ) < 20:

                duplicate_id_samples.append(
                    company_id
                )

        else:

            seen_ids.add(
                company_id
            )


    if chunk_number % 5 == 0:

        print(
            f"Checked chunks: {chunk_number}"
        )


print("\nID AUDIT")

print(
    "Unique IDs:",
    len(seen_ids)
)

print(
    "Missing IDs:",
    missing_id_count
)

print(
    "Duplicate ID rows:",
    duplicate_id_count
)

print(
    "Duplicate ID samples:",
    duplicate_id_samples
)

Checked chunks: 5
Checked chunks: 10
Checked chunks: 15
Checked chunks: 20
Checked chunks: 25

ID AUDIT
Unique IDs: 2807490
Missing IDs: 0
Duplicate ID rows: 2
Duplicate ID samples: ['dff36e62-25ce-412e-a4b8-6b0dfce1288d', '9bd7c4a1-c3b4-4eaf-b2ce-72d2b166a08f']


In [10]:
# CELL 7 - INSPECT IMPORTANT CATEGORICAL VALUES

categorical_columns = [
    "operating_status",
    "company_type",
    "ipo_status",
    "last_funding_type",
    "num_employees_enum"
]


categorical_value_sets = {
    column: set()
    for column in categorical_columns
}


for chunk_number, chunk in enumerate(
    pd.read_csv(
        LARGE_DATASET_FILE,
        usecols=categorical_columns,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    for column in categorical_columns:

        values = (
            chunk[
                column
            ]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()
        )


        categorical_value_sets[
            column
        ].update(values)


print("CATEGORICAL VALUES")


for column in categorical_columns:

    values = sorted(
        categorical_value_sets[
            column
        ]
    )

    print(
        f"\n{column}"
    )

    print(
        "Unique values:",
        len(values)
    )

    print(
        values[:100]
    )

CATEGORICAL VALUES

operating_status
Unique values: 2
['active', 'closed']

company_type
Unique values: 2
['for_profit', 'non_profit']

ipo_status
Unique values: 3
['delisted', 'private', 'public']

last_funding_type
Unique values: 28
['angel', 'convertible_note', 'corporate_round', 'debt_financing', 'equity_crowdfunding', 'grant', 'initial_coin_offering', 'non_equity_assistance', 'post_ipo_debt', 'post_ipo_equity', 'post_ipo_secondary', 'pre_seed', 'private_equity', 'product_crowdfunding', 'secondary_market', 'seed', 'series_a', 'series_b', 'series_c', 'series_d', 'series_e', 'series_f', 'series_g', 'series_h', 'series_i', 'series_j', 'series_unknown', 'undisclosed']

num_employees_enum
Unique values: 9
['c_00001_00010', 'c_00011_00050', 'c_00051_00100', 'c_00101_00250', 'c_00251_00500', 'c_00501_01000', 'c_01001_05000', 'c_05001_10000', 'c_10001_max']


In [11]:
# CELL 8 - INSPECT DUPLICATE ID ROWS

duplicate_ids = [
    "dff36e62-25ce-412e-a4b8-6b0dfce1288d",
    "9bd7c4a1-c3b4-4eaf-b2ce-72d2b166a08f"
]


duplicate_rows = []


for chunk in pd.read_csv(
    LARGE_DATASET_FILE,
    chunksize=CHUNK_SIZE,
    low_memory=False
):

    matches = (
        chunk[
            chunk["id"].isin(
                duplicate_ids
            )
        ]
        .copy()
    )

    if len(matches) > 0:
        duplicate_rows.append(
            matches
        )


duplicate_id_rows_df = pd.concat(
    duplicate_rows,
    ignore_index=True
)


print(
    "Duplicate ID rows found:",
    len(duplicate_id_rows_df)
)


display(
    duplicate_id_rows_df[
        [
            "id",
            "name",
            "permalink",
            "website",
            "operating_status",
            "company_type",
            "ipo_status",
            "created_at"
        ]
    ]
    .sort_values(
        [
            "id",
            "created_at"
        ]
    )
)

Duplicate ID rows found: 4


,id,name,permalink,website,operating_status,company_type,ipo_status,created_at
0,9bd7c4a1-c3b4-4eaf-b2ce-72d2b166a08f,Global Healthcare Solutions,global-healthcare-solutions,https://www.ghs-care.co.uk,active,for_profit,private,2024-08-18 10:06:26.992004+00
3,9bd7c4a1-c3b4-4eaf-b2ce-72d2b166a08f,Global Healthcare Solutions,global-healthcare-solutions,https://www.ghs-care.co.uk,active,for_profit,private,2024-08-18 10:06:26.992004+00
1,dff36e62-25ce-412e-a4b8-6b0dfce1288d,Budgify,budgify-288d,NaN,active,for_profit,private,2024-08-15 07:55:07.325505+00
2,dff36e62-25ce-412e-a4b8-6b0dfce1288d,Budgify,budgify-288d,NaN,active,for_profit,private,2024-08-15 07:55:07.325505+00


In [12]:
# CELL 9 - COMPARE DUPLICATE ID RECORDS

duplicate_comparison_rows = []


for company_id, group in (
    duplicate_id_rows_df
    .groupby("id")
):

    group = group.reset_index(
        drop=True
    )

    record_count = len(group)

    differing_columns = []

    if record_count > 1:

        for column in group.columns:

            values = (
                group[column]
                .astype("string")
                .fillna("<NA>")
                .unique()
            )

            if len(values) > 1:
                differing_columns.append(
                    column
                )


    duplicate_comparison_rows.append(
        {
            "id": company_id,
            "record_count": record_count,
            "differing_column_count":
                len(differing_columns),
            "differing_columns":
                differing_columns
        }
    )


duplicate_id_comparison_df = (
    pd.DataFrame(
        duplicate_comparison_rows
    )
)


display(
    duplicate_id_comparison_df
)

,id,record_count,differing_column_count,differing_columns
0,9bd7c4a1-c3b4-4eaf-b2ce-72d2b166a08f,2,0,[]
1,dff36e62-25ce-412e-a4b8-6b0dfce1288d,2,0,[]


In [13]:
# CELL 10 - DEFINE EMPLOYEE RANGE MAPPING

employee_range_mapping = {
    "c_00001_00010": (1, 10),
    "c_00011_00050": (11, 50),
    "c_00051_00100": (51, 100),
    "c_00101_00250": (101, 250),
    "c_00251_00500": (251, 500),
    "c_00501_01000": (501, 1000),
    "c_01001_05000": (1001, 5000),
    "c_05001_10000": (5001, 10000),
    "c_10001_max": (10001, None),
}


employee_mapping_df = pd.DataFrame(
    [
        {
            "num_employees_enum": key,
            "employee_min": value[0],
            "employee_max": value[1],
            "employee_midpoint": (
                (
                    value[0]
                    + value[1]
                ) / 2
                if value[1] is not None
                else np.nan
            )
        }
        for key, value
        in employee_range_mapping.items()
    ]
)


display(
    employee_mapping_df
)

,num_employees_enum,employee_min,employee_max,employee_midpoint
0,c_00001_00010,1,10.0,5.5
1,c_00011_00050,11,50.0,30.5
2,c_00051_00100,51,100.0,75.5
3,c_00101_00250,101,250.0,175.5
4,c_00251_00500,251,500.0,375.5
5,c_00501_01000,501,1000.0,750.5
6,c_01001_05000,1001,5000.0,3000.5
7,c_05001_10000,5001,10000.0,7500.5
8,c_10001_max,10001,NaN,NaN


In [14]:
# CELL 11 - DEFINE COLUMNS FOR PROCESSED COMPANY DATASET

processed_columns = [
    # Identity / profile
    "id",
    "created_at",
    "name",
    "short_description",
    "permalink",
    "url",
    "website",

    # Company status
    "operating_status",
    "company_type",
    "ipo_status",
    "founded_on",

    # Business context
    "categories",
    "locations",
    "founders",

    # Employee size
    "num_employees_enum",

    # Funding
    "funding_total",
    "num_funding_rounds",
    "last_funding_type",
    "last_funding_at",
    "num_investors",
    "num_lead_investors",

    # Web / app traction
    "semrush_global_rank",
    "semrush_visits_latest_month",
    "apptopia_total_apps",
    "apptopia_total_downloads",

    # Growth / investor insights
    "growth_insight_description",
    "growth_insight_indicator",
    "growth_insight_direction",
    "growth_insight_confidence",
    "investor_insight_description",
]


missing_processed_columns = [
    column
    for column in processed_columns
    if column not in sample_df.columns
]


print(
    "Selected columns:",
    len(processed_columns)
)

print(
    "Missing selected columns:",
    missing_processed_columns
)

print("\nSELECTED COLUMN LIST")

for column in processed_columns:
    print(column)

Selected columns: 30
Missing selected columns: []

SELECTED COLUMN LIST
id
created_at
name
short_description
permalink
url
website
operating_status
company_type
ipo_status
founded_on
categories
locations
founders
num_employees_enum
funding_total
num_funding_rounds
last_funding_type
last_funding_at
num_investors
num_lead_investors
semrush_global_rank
semrush_visits_latest_month
apptopia_total_apps
apptopia_total_downloads
growth_insight_description
growth_insight_indicator
growth_insight_direction
growth_insight_confidence
investor_insight_description


In [15]:
# CELL 12 - DEFINE CLEANING HELPERS

def clean_text_series(series):
    return (
        series
        .astype("string")
        .str.strip()
        .replace(
            {
                "": pd.NA,
                "nan": pd.NA,
                "None": pd.NA,
                "null": pd.NA
            }
        )
    )


def add_employee_range_columns(df):

    df = df.copy()

    df["employee_min"] = (
        df["num_employees_enum"]
        .map(
            {
                key: value[0]
                for key, value
                in employee_range_mapping.items()
            }
        )
    )

    df["employee_max"] = (
        df["num_employees_enum"]
        .map(
            {
                key: value[1]
                for key, value
                in employee_range_mapping.items()
            }
        )
    )

    df["employee_midpoint"] = np.where(
        df["employee_max"].notna(),
        (
            df["employee_min"]
            + df["employee_max"]
        ) / 2,
        np.nan
    )

    return df

In [16]:
# CELL 13 - TEST CLEANING ON SMALL SAMPLE

processed_sample_df = (
    sample_df[
        processed_columns
    ]
    .copy()
)


text_columns = [
    "id",
    "name",
    "short_description",
    "permalink",
    "url",
    "website",
    "operating_status",
    "company_type",
    "ipo_status",
    "categories",
    "locations",
    "founders",
    "num_employees_enum",
    "last_funding_type",
    "growth_insight_description",
    "growth_insight_indicator",
    "growth_insight_direction",
    "investor_insight_description",
]


for column in text_columns:

    processed_sample_df[
        column
    ] = clean_text_series(
        processed_sample_df[
            column
        ]
    )


processed_sample_df = (
    add_employee_range_columns(
        processed_sample_df
    )
)


processed_sample_df = (
    processed_sample_df
    .drop_duplicates(
        subset=["id"],
        keep="first"
    )
    .reset_index(drop=True)
)


print(
    "Processed sample rows:",
    len(processed_sample_df)
)

print(
    "Processed columns:",
    len(processed_sample_df.columns)
)


display(
    processed_sample_df.head(10)
)

Processed sample rows: 100
Processed columns: 33


,id,created_at,name,short_description,permalink,url,website,operating_status,company_type,ipo_status,...,apptopia_total_apps,apptopia_total_downloads,growth_insight_description,growth_insight_indicator,growth_insight_direction,growth_insight_confidence,investor_insight_description,employee_min,employee_max,employee_midpoint
0,a8de17a2-5700-40d0-8331-9f8fa7b3df27,2024-08-18 09:47:54.407151+00,GivingTech,"GivingTech - Giving Technologies, Inc is creat...",𝗚𝗶𝘃𝗶𝗻𝗴𝗧𝗲𝗰h,https://www.crunchbase.com/organization/𝗚𝗶𝘃𝗶𝗻𝗴...,https://giving.technology,active,for_profit,private,...,NaN,NaN,<NA>,<NA>,<NA>,NaN,<NA>,1.0,10.0,5.5
1,dfa41d55-72f7-4745-8ba9-a1fe664e90f8,2024-08-24 18:14:46.291141+00,X+,X+ is an elite club for DeGods whales and buil...,𝐗,https://www.crunchbase.com/organization/𝐗,<NA>,active,<NA>,<NA>,...,NaN,NaN,<NA>,<NA>,<NA>,NaN,<NA>,NaN,NaN,NaN
2,5d7e4faf-7880-47eb-bd72-902057733280,2024-08-18 06:49:06.951712+00,F. Rego,"F. Rego offers insurance consulting, brokerage...",𝗙-𝗥𝗲𝗴𝗼,https://www.crunchbase.com/organization/𝗙-𝗥𝗲𝗴𝗼,https://frego.pt,active,for_profit,private,...,NaN,NaN,<NA>,<NA>,<NA>,NaN,<NA>,51.0,100.0,75.5
3,075b9784-a35f-4682-9422-412e6791bde0,2024-08-20 10:20:19.463579+00,O’dara Exotic Skincare,O’dara Exotic Skincare is a skincare line that...,𝐎-𝐝𝐚𝐫𝐚-𝐄𝐱𝐨𝐭𝐢𝐜-𝐒𝐤𝐢𝐧𝐜𝐚𝐫e,https://www.crunchbase.com/organization/𝐎-𝐝𝐚𝐫𝐚...,https://www.odaraexoticskin.com,active,for_profit,private,...,NaN,NaN,<NA>,<NA>,<NA>,NaN,<NA>,NaN,NaN,NaN
4,ee90241c-d768-4167-a428-feefda59da90,2024-08-24 17:07:32.901299+00,!Bewust sociaal op web,!Bewust sociaal op web is providing social med...,bewust-sociaal-op-web,https://www.crunchbase.com/organization/bewust...,https://bewustsociaalopweb.nl,active,for_profit,private,...,NaN,NaN,<NA>,<NA>,<NA>,NaN,<NA>,1.0,10.0,5.5
5,9a0d9dd1-8024-41f7-984b-45ac32997339,2024-08-17 09:52:46.624066+00,!Creatice,!Creatice is a Education based company.,creatice,https://www.crunchbase.com/organization/creatice,http://feiquchuangzao.com,closed,for_profit,private,...,NaN,NaN,<NA>,<NA>,<NA>,NaN,<NA>,11.0,50.0,30.5
6,bce54cde-ca3b-41be-81b9-c9a1d01e4e8c,2024-08-17 15:57:48.866428+00,!DOEVE*,"!DOEVE* offers marketing, communication and sa...",doeve,https://www.crunchbase.com/organization/doeve,http://www.doeve.nl/,active,for_profit,private,...,NaN,NaN,<NA>,<NA>,<NA>,NaN,<NA>,1.0,10.0,5.5
7,d4b0e8f9-86f2-022d-6d9a-598f1a7033cf,2024-08-18 03:37:00.299812+00,!FEST,Fest is a company that creates a unique atmosp...,fest,https://www.crunchbase.com/organization/fest,http://www.fest.lviv.ua,active,for_profit,private,...,2.0,11354.0,<NA>,<NA>,<NA>,NaN,<NA>,101.0,250.0,175.5
8,8e2c2fbd-2244-32eb-8e35-9ea9cefeb90f,2024-08-19 07:52:08.916809+00,!K7,A music video production company based in Berlin.,k7-2,https://www.crunchbase.com/organization/k7-2,http://www.k7.com,active,for_profit,private,...,NaN,NaN,<NA>,<NA>,<NA>,NaN,<NA>,11.0,50.0,30.5
9,7268c6ad-9c37-3265-7545-0f4c07fe22a2,2024-08-22 14:38:10.198084+00,!SHOUTTAG,A !shouttag is a powerful call to action that ...,shouttag,https://www.crunchbase.com/organization/shouttag,https://shouttag.com/!shouttag,active,for_profit,private,...,1.0,NaN,<NA>,<NA>,<NA>,NaN,<NA>,1.0,10.0,5.5


In [17]:
# CELL 14 - FULL CHUNKED CLEANING AND SAVE PROCESSED DATASET

PROCESSED_DATASET_FILE = Path(
    "data/company_dataset/processed/company_dataset_processed.csv"
)

CHUNK_SIZE = 100_000


# Remove old output if it already exists
if PROCESSED_DATASET_FILE.exists():
    PROCESSED_DATASET_FILE.unlink()


seen_ids_full = set()

processed_row_count = 0
duplicate_rows_removed = 0

first_write = True


for chunk_number, chunk in enumerate(
    pd.read_csv(
        LARGE_DATASET_FILE,
        usecols=processed_columns,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    # -----------------------------
    # Clean text columns
    # -----------------------------

    for column in text_columns:

        if column in chunk.columns:

            chunk[column] = clean_text_series(
                chunk[column]
            )


    # -----------------------------
    # Remove duplicate IDs globally
    # -----------------------------

    duplicate_mask = (
        chunk["id"]
        .astype("string")
        .isin(seen_ids_full)
    )

    duplicate_rows_removed += (
        duplicate_mask.sum()
    )

    chunk = (
        chunk[
            ~duplicate_mask
        ]
        .copy()
    )


    current_ids = (
        chunk[
            "id"
        ]
        .dropna()
        .astype(str)
        .tolist()
    )

    seen_ids_full.update(
        current_ids
    )


    # -----------------------------
    # Employee range enrichment
    # -----------------------------

    chunk = add_employee_range_columns(
        chunk
    )


    # -----------------------------
    # Append to processed CSV
    # -----------------------------

    chunk.to_csv(
        PROCESSED_DATASET_FILE,
        mode="w" if first_write else "a",
        header=first_write,
        index=False
    )


    first_write = False

    processed_row_count += len(chunk)


    print(
        f"Processed chunk {chunk_number} "
        f"| total rows written: {processed_row_count:,}"
    )


print("\nFULL PROCESSING COMPLETE")

print(
    "Rows written:",
    processed_row_count
)

print(
    "Duplicate rows removed:",
    duplicate_rows_removed
)

print(
    "Unique IDs tracked:",
    len(seen_ids_full)
)

print(
    "Output file:",
    PROCESSED_DATASET_FILE
)

Processed chunk 1 | total rows written: 100,000
Processed chunk 2 | total rows written: 200,000
Processed chunk 3 | total rows written: 300,000
Processed chunk 4 | total rows written: 400,000
Processed chunk 5 | total rows written: 500,000
Processed chunk 6 | total rows written: 600,000
Processed chunk 7 | total rows written: 700,000
Processed chunk 8 | total rows written: 800,000
Processed chunk 9 | total rows written: 900,000
Processed chunk 10 | total rows written: 1,000,000
Processed chunk 11 | total rows written: 1,100,000
Processed chunk 12 | total rows written: 1,200,000
Processed chunk 13 | total rows written: 1,300,000
Processed chunk 14 | total rows written: 1,400,000
Processed chunk 15 | total rows written: 1,500,000
Processed chunk 16 | total rows written: 1,600,000
Processed chunk 17 | total rows written: 1,700,000
Processed chunk 18 | total rows written: 1,800,000
Processed chunk 19 | total rows written: 1,900,000
Processed chunk 20 | total rows written: 2,000,000
Process

In [18]:
# CELL 15 - AUDIT PROCESSED DATASET

processed_total_rows = 0

processed_missing_counts = None


for chunk_number, chunk in enumerate(
    pd.read_csv(
        PROCESSED_DATASET_FILE,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    processed_total_rows += len(chunk)

    chunk_missing = (
        chunk
        .isna()
        .sum()
    )


    if processed_missing_counts is None:

        processed_missing_counts = (
            chunk_missing.copy()
        )

    else:

        processed_missing_counts = (
            processed_missing_counts
            .add(
                chunk_missing,
                fill_value=0
            )
        )


print(
    "Processed total rows:",
    processed_total_rows
)


processed_quality_df = pd.DataFrame(
    {
        "column":
            processed_missing_counts.index,

        "missing_count":
            processed_missing_counts.values
    }
)


processed_quality_df[
    "missing_pct"
] = (
    processed_quality_df[
        "missing_count"
    ]
    /
    processed_total_rows
    *
    100
)


processed_quality_df = (
    processed_quality_df
    .sort_values(
        "missing_pct",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    processed_quality_df
)

Processed total rows: 2807491


,column,missing_count,missing_pct
0,growth_insight_direction,2773719,98.797075
1,growth_insight_confidence,2773719,98.797075
2,investor_insight_description,2773173,98.777627
3,apptopia_total_downloads,2754305,98.105568
4,growth_insight_description,2739198,97.567472
5,growth_insight_indicator,2739198,97.567472
6,num_lead_investors,2659639,94.733661
7,apptopia_total_apps,2617803,93.243505
8,funding_total,2586712,92.136075
9,num_investors,2577282,91.800187


In [19]:
# CELL 16 - FIND REMAINING DUPLICATE ID IN PROCESSED DATASET

known_duplicate_ids = [
    "dff36e62-25ce-412e-a4b8-6b0dfce1288d",
    "9bd7c4a1-c3b4-4eaf-b2ce-72d2b166a08f",
]


remaining_duplicate_rows = []


for chunk in pd.read_csv(
    PROCESSED_DATASET_FILE,
    usecols=[
        "id",
        "name",
        "permalink",
        "created_at"
    ],
    chunksize=CHUNK_SIZE,
    low_memory=False
):

    matches = chunk[
        chunk["id"].isin(
            known_duplicate_ids
        )
    ]

    if len(matches) > 0:
        remaining_duplicate_rows.append(
            matches.copy()
        )


remaining_duplicate_rows_df = pd.concat(
    remaining_duplicate_rows,
    ignore_index=True
)


print("ROWS FOR KNOWN DUPLICATE IDS")

display(
    remaining_duplicate_rows_df
    .sort_values(
        [
            "id",
            "created_at"
        ]
    )
)


print("\nCOUNTS")

print(
    remaining_duplicate_rows_df[
        "id"
    ].value_counts()
)

ROWS FOR KNOWN DUPLICATE IDS


,id,created_at,name,permalink
0,9bd7c4a1-c3b4-4eaf-b2ce-72d2b166a08f,2024-08-18 10:06:26.992004+00,Global Healthcare Solutions,global-healthcare-solutions
1,dff36e62-25ce-412e-a4b8-6b0dfce1288d,2024-08-15 07:55:07.325505+00,Budgify,budgify-288d
2,dff36e62-25ce-412e-a4b8-6b0dfce1288d,2024-08-15 07:55:07.325505+00,Budgify,budgify-288d



COUNTS
id
dff36e62-25ce-412e-a4b8-6b0dfce1288d    2
9bd7c4a1-c3b4-4eaf-b2ce-72d2b166a08f    1
Name: count, dtype: int64


In [20]:
# CELL 17 - REMOVE THE LAST DUPLICATE FROM PROCESSED DATASET

PROCESSED_TEMP_FILE = Path(
    "data/company_dataset/processed/company_dataset_processed_deduped.csv"
)


if PROCESSED_TEMP_FILE.exists():
    PROCESSED_TEMP_FILE.unlink()


seen_ids = set()
rows_written = 0
duplicates_removed = 0
first_write = True


for chunk_number, chunk in enumerate(
    pd.read_csv(
        PROCESSED_DATASET_FILE,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    # IMPORTANT:
    # First remove duplicates INSIDE the current chunk.
    before_chunk_dedup = len(chunk)

    chunk = (
        chunk
        .drop_duplicates(
            subset=["id"],
            keep="first"
        )
        .copy()
    )

    duplicates_removed += (
        before_chunk_dedup
        - len(chunk)
    )


    # Then remove IDs already seen in previous chunks.
    previous_chunk_duplicate_mask = (
        chunk["id"]
        .astype(str)
        .isin(seen_ids)
    )

    duplicates_removed += (
        previous_chunk_duplicate_mask.sum()
    )

    chunk = (
        chunk[
            ~previous_chunk_duplicate_mask
        ]
        .copy()
    )


    seen_ids.update(
        chunk["id"]
        .dropna()
        .astype(str)
        .tolist()
    )


    chunk.to_csv(
        PROCESSED_TEMP_FILE,
        mode="w" if first_write else "a",
        header=first_write,
        index=False
    )

    first_write = False
    rows_written += len(chunk)


    print(
        f"Chunk {chunk_number} "
        f"| rows written: {rows_written:,}"
    )


print("\nDEDUP COMPLETE")

print(
    "Rows written:",
    rows_written
)

print(
    "Duplicates removed:",
    duplicates_removed
)

print(
    "Unique IDs:",
    len(seen_ids)
)

print(
    "Output:",
    PROCESSED_TEMP_FILE
)

Chunk 1 | rows written: 100,000
Chunk 2 | rows written: 200,000
Chunk 3 | rows written: 300,000
Chunk 4 | rows written: 400,000
Chunk 5 | rows written: 500,000
Chunk 6 | rows written: 600,000
Chunk 7 | rows written: 700,000
Chunk 8 | rows written: 800,000
Chunk 9 | rows written: 900,000
Chunk 10 | rows written: 1,000,000
Chunk 11 | rows written: 1,100,000
Chunk 12 | rows written: 1,200,000
Chunk 13 | rows written: 1,300,000
Chunk 14 | rows written: 1,400,000
Chunk 15 | rows written: 1,500,000
Chunk 16 | rows written: 1,600,000
Chunk 17 | rows written: 1,700,000
Chunk 18 | rows written: 1,800,000
Chunk 19 | rows written: 1,900,000
Chunk 20 | rows written: 2,000,000
Chunk 21 | rows written: 2,100,000
Chunk 22 | rows written: 2,199,999
Chunk 23 | rows written: 2,299,999
Chunk 24 | rows written: 2,399,999
Chunk 25 | rows written: 2,499,999
Chunk 26 | rows written: 2,599,999
Chunk 27 | rows written: 2,699,999
Chunk 28 | rows written: 2,799,999
Chunk 29 | rows written: 2,807,490

DEDUP COMPL

In [21]:
# CELL 18 - SET FINAL PROCESSED SOURCE DATASET

PROCESSED_DEDUP_FILE = Path(
    "data/company_dataset/processed/company_dataset_processed_deduped.csv"
)


print(
    "Processed dedup file:",
    PROCESSED_DEDUP_FILE
)

print(
    "Exists:",
    PROCESSED_DEDUP_FILE.exists()
)


if PROCESSED_DEDUP_FILE.exists():

    print(
        "Size MB:",
        round(
            PROCESSED_DEDUP_FILE.stat().st_size
            / 1024
            / 1024,
            2
        )
    )


processed_sample_df = pd.read_csv(
    PROCESSED_DEDUP_FILE,
    nrows=100,
    low_memory=False
)


print(
    "\nSample rows:",
    len(processed_sample_df)
)

print(
    "Columns:",
    len(processed_sample_df.columns)
)

print("\nCOLUMN LIST")

print(
    processed_sample_df.columns.tolist()
)

Processed dedup file: data\company_dataset\company_dataset_processed_deduped.csv
Exists: True
Size MB: 1308.9

Sample rows: 100
Columns: 33

COLUMN LIST
['id', 'created_at', 'name', 'short_description', 'semrush_global_rank', 'semrush_visits_latest_month', 'num_investors', 'funding_total', 'num_funding_rounds', 'last_funding_type', 'last_funding_at', 'apptopia_total_apps', 'apptopia_total_downloads', 'num_lead_investors', 'company_type', 'operating_status', 'founded_on', 'categories', 'founders', 'website', 'ipo_status', 'num_employees_enum', 'locations', 'growth_insight_description', 'growth_insight_indicator', 'growth_insight_direction', 'growth_insight_confidence', 'investor_insight_description', 'permalink', 'url', 'employee_min', 'employee_max', 'employee_midpoint']


In [22]:
# CELL 19 - DEFINE FEATURE ENGINEERING HELPERS

REFERENCE_YEAR = 2026


def add_company_features(df):

    df = df.copy()


    # -----------------------------------
    # Founded date / company age
    # -----------------------------------

    df[
        "founded_on"
    ] = pd.to_datetime(
        df[
            "founded_on"
        ],
        errors="coerce"
    )


    df[
        "founded_year"
    ] = (
        df[
            "founded_on"
        ]
        .dt.year
    )


    df[
        "company_age_years"
    ] = np.where(
        df[
            "founded_year"
        ].notna(),
        REFERENCE_YEAR
        - df[
            "founded_year"
        ],
        np.nan
    )


    # Negative age = invalid/future founded date
    df.loc[
        df[
            "company_age_years"
        ] < 0,
        "company_age_years"
    ] = np.nan


    # -----------------------------------
    # Funding numeric cleanup
    # -----------------------------------

    numeric_columns = [
        "funding_total",
        "num_funding_rounds",
        "num_investors",
        "num_lead_investors",
        "semrush_global_rank",
        "semrush_visits_latest_month",
        "apptopia_total_apps",
        "apptopia_total_downloads",
        "employee_min",
        "employee_max",
        "employee_midpoint",
        "growth_insight_confidence",
    ]


    for column in numeric_columns:

        if column in df.columns:

            df[
                column
            ] = pd.to_numeric(
                df[
                    column
                ],
                errors="coerce"
            )


    # -----------------------------------
    # Funding intensity
    # -----------------------------------

    df[
        "funding_per_round"
    ] = np.where(
        (
            df[
                "funding_total"
            ].notna()
        )
        &
        (
            df[
                "num_funding_rounds"
            ].notna()
        )
        &
        (
            df[
                "num_funding_rounds"
            ] > 0
        ),
        df[
            "funding_total"
        ]
        /
        df[
            "num_funding_rounds"
        ],
        np.nan
    )


    # -----------------------------------
    # Presence / signal flags
    # -----------------------------------

    df[
        "has_funding_data"
    ] = (
        df[
            "funding_total"
        ].notna()
        |
        df[
            "num_funding_rounds"
        ].notna()
    )


    df[
        "has_investor_data"
    ] = (
        df[
            "num_investors"
        ].notna()
        |
        df[
            "num_lead_investors"
        ].notna()
    )


    df[
        "has_web_traction"
    ] = (
        df[
            "semrush_global_rank"
        ].notna()
        |
        df[
            "semrush_visits_latest_month"
        ].notna()
    )


    df[
        "has_app_traction"
    ] = (
        df[
            "apptopia_total_apps"
        ].notna()
        |
        df[
            "apptopia_total_downloads"
        ].notna()
    )


    df[
        "has_growth_insight"
    ] = (
        df[
            "growth_insight_description"
        ].notna()
    )


    df[
        "has_investor_insight"
    ] = (
        df[
            "investor_insight_description"
        ].notna()
    )


    return df

In [23]:
# CELL 20 - TEST FEATURE ENGINEERING ON SAMPLE

feature_sample_df = (
    add_company_features(
        processed_sample_df
    )
)


feature_columns = [
    "id",
    "name",
    "founded_on",
    "founded_year",
    "company_age_years",
    "num_employees_enum",
    "employee_min",
    "employee_max",
    "employee_midpoint",
    "funding_total",
    "num_funding_rounds",
    "funding_per_round",
    "num_investors",
    "num_lead_investors",
    "has_funding_data",
    "has_investor_data",
    "has_web_traction",
    "has_app_traction",
    "has_growth_insight",
    "has_investor_insight",
]


print(
    "Feature sample rows:",
    len(feature_sample_df)
)

print(
    "Feature sample columns:",
    len(feature_sample_df.columns)
)


display(
    feature_sample_df[
        feature_columns
    ].head(20)
)

Feature sample rows: 100
Feature sample columns: 42


,id,name,founded_on,founded_year,company_age_years,num_employees_enum,employee_min,employee_max,employee_midpoint,funding_total,num_funding_rounds,funding_per_round,num_investors,num_lead_investors,has_funding_data,has_investor_data,has_web_traction,has_app_traction,has_growth_insight,has_investor_insight
0,a8de17a2-5700-40d0-8331-9f8fa7b3df27,GivingTech,2020-01-01,2020.0,6.0,c_00001_00010,1.0,10.0,5.5,NaN,NaN,NaN,NaN,NaN,False,False,True,False,False,False
1,dfa41d55-72f7-4745-8ba9-a1fe664e90f8,X+,2022-01-01,2022.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,False,False,False
2,5d7e4faf-7880-47eb-bd72-902057733280,F. Rego,1979-01-01,1979.0,47.0,c_00051_00100,51.0,100.0,75.5,NaN,NaN,NaN,NaN,NaN,False,False,True,False,False,False
3,075b9784-a35f-4682-9422-412e6791bde0,O’dara Exotic Skincare,NaT,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,0.0,1.0,1.0,True,True,False,False,False,False
4,ee90241c-d768-4167-a428-feefda59da90,!Bewust sociaal op web,2019-01-01,2019.0,7.0,c_00001_00010,1.0,10.0,5.5,NaN,NaN,NaN,NaN,NaN,False,False,False,False,False,False
5,9a0d9dd1-8024-41f7-984b-45ac32997339,!Creatice,2018-01-01,2018.0,8.0,c_00011_00050,11.0,50.0,30.5,302000.0,1.0,302000.0,NaN,NaN,True,False,False,False,False,False
6,bce54cde-ca3b-41be-81b9-c9a1d01e4e8c,!DOEVE*,2014-01-01,2014.0,12.0,c_00001_00010,1.0,10.0,5.5,NaN,NaN,NaN,NaN,NaN,False,False,False,False,False,False
7,d4b0e8f9-86f2-022d-6d9a-598f1a7033cf,!FEST,2007-01-01,2007.0,19.0,c_00101_00250,101.0,250.0,175.5,NaN,NaN,NaN,NaN,NaN,False,False,True,True,False,False
8,8e2c2fbd-2244-32eb-8e35-9ea9cefeb90f,!K7,1985-01-01,1985.0,41.0,c_00011_00050,11.0,50.0,30.5,NaN,NaN,NaN,NaN,NaN,False,False,True,False,False,False
9,7268c6ad-9c37-3265-7545-0f4c07fe22a2,!SHOUTTAG,2012-02-17,2012.0,14.0,c_00001_00010,1.0,10.0,5.5,NaN,NaN,NaN,NaN,NaN,False,False,False,True,False,False


In [24]:
# CELL 21 - FULL CHUNKED FEATURE ENGINEERING

FEATURE_DATASET_FILE = Path(
    "data/company_dataset/processed/company_dataset_features.csv"
)

FEATURE_CHUNK_SIZE = 100_000


if FEATURE_DATASET_FILE.exists():
    FEATURE_DATASET_FILE.unlink()


feature_rows_written = 0
first_write = True


for chunk_number, chunk in enumerate(
    pd.read_csv(
        PROCESSED_DEDUP_FILE,
        chunksize=FEATURE_CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    feature_chunk = add_company_features(
        chunk
    )


    feature_chunk.to_csv(
        FEATURE_DATASET_FILE,
        mode="w" if first_write else "a",
        header=first_write,
        index=False
    )


    first_write = False

    feature_rows_written += len(
        feature_chunk
    )


    print(
        f"Processed chunk {chunk_number} "
        f"| total rows written: "
        f"{feature_rows_written:,}"
    )


print("\nFEATURE ENGINEERING COMPLETE")

print(
    "Rows written:",
    feature_rows_written
)

print(
    "Output:",
    FEATURE_DATASET_FILE
)

Processed chunk 1 | total rows written: 100,000
Processed chunk 2 | total rows written: 200,000
Processed chunk 3 | total rows written: 300,000
Processed chunk 4 | total rows written: 400,000
Processed chunk 5 | total rows written: 500,000
Processed chunk 6 | total rows written: 600,000
Processed chunk 7 | total rows written: 700,000
Processed chunk 8 | total rows written: 800,000
Processed chunk 9 | total rows written: 900,000
Processed chunk 10 | total rows written: 1,000,000
Processed chunk 11 | total rows written: 1,100,000
Processed chunk 12 | total rows written: 1,200,000
Processed chunk 13 | total rows written: 1,300,000
Processed chunk 14 | total rows written: 1,400,000
Processed chunk 15 | total rows written: 1,500,000
Processed chunk 16 | total rows written: 1,600,000
Processed chunk 17 | total rows written: 1,700,000
Processed chunk 18 | total rows written: 1,800,000
Processed chunk 19 | total rows written: 1,900,000
Processed chunk 20 | total rows written: 2,000,000
Process

In [25]:
# CELL 22 - AUDIT FULL FEATURE DATASET

feature_total_rows = 0

feature_stats = {
    "founded_year_available": 0,
    "company_age_available": 0,
    "employee_data_available": 0,
    "funding_data_available": 0,
    "investor_data_available": 0,
    "web_traction_available": 0,
    "app_traction_available": 0,
    "growth_insight_available": 0,
    "investor_insight_available": 0,
}


for chunk in pd.read_csv(
    FEATURE_DATASET_FILE,
    chunksize=FEATURE_CHUNK_SIZE,
    low_memory=False
):

    feature_total_rows += len(
        chunk
    )


    feature_stats[
        "founded_year_available"
    ] += (
        chunk[
            "founded_year"
        ].notna().sum()
    )


    feature_stats[
        "company_age_available"
    ] += (
        chunk[
            "company_age_years"
        ].notna().sum()
    )


    feature_stats[
        "employee_data_available"
    ] += (
        chunk[
            "employee_min"
        ].notna().sum()
    )


    feature_stats[
        "funding_data_available"
    ] += (
        chunk[
            "has_funding_data"
        ].fillna(False).astype(bool).sum()
    )


    feature_stats[
        "investor_data_available"
    ] += (
        chunk[
            "has_investor_data"
        ].fillna(False).astype(bool).sum()
    )


    feature_stats[
        "web_traction_available"
    ] += (
        chunk[
            "has_web_traction"
        ].fillna(False).astype(bool).sum()
    )


    feature_stats[
        "app_traction_available"
    ] += (
        chunk[
            "has_app_traction"
        ].fillna(False).astype(bool).sum()
    )


    feature_stats[
        "growth_insight_available"
    ] += (
        chunk[
            "has_growth_insight"
        ].fillna(False).astype(bool).sum()
    )


    feature_stats[
        "investor_insight_available"
    ] += (
        chunk[
            "has_investor_insight"
        ].fillna(False).astype(bool).sum()
    )


print(
    "Feature total rows:",
    feature_total_rows
)


feature_availability_df = pd.DataFrame(
    [
        {
            "feature": feature,
            "available_count": count,
            "available_pct": (
                count
                / feature_total_rows
                * 100
            )
        }
        for feature, count
        in feature_stats.items()
    ]
)


display(
    feature_availability_df
    .sort_values(
        "available_pct",
        ascending=False
    )
)

Feature total rows: 2807490


,feature,available_count,available_pct
2,employee_data_available,2370227,84.425127
0,founded_year_available,2322078,82.710108
1,company_age_available,2322078,82.710108
5,web_traction_available,1118067,39.824434
3,funding_data_available,290204,10.336778
4,investor_data_available,230209,8.199815
6,app_traction_available,191174,6.809428
7,growth_insight_available,68293,2.432529
8,investor_insight_available,34318,1.222373


In [26]:
# CELL 23 - INSPECT CATEGORY AND LOCATION FORMATS

category_location_sample_df = pd.read_csv(
    FEATURE_DATASET_FILE,
    usecols=[
        "id",
        "name",
        "categories",
        "locations"
    ],
    nrows=200,
    low_memory=False
)


print("CATEGORY EXAMPLES")

display(
    category_location_sample_df[
        [
            "name",
            "categories"
        ]
    ]
    .dropna(
        subset=["categories"]
    )
    .head(30)
)


print("\nLOCATION EXAMPLES")

display(
    category_location_sample_df[
        [
            "name",
            "locations"
        ]
    ]
    .dropna(
        subset=["locations"]
    )
    .head(30)
)

CATEGORY EXAMPLES


,name,categories
2,F. Rego,"Financial Services, Insurance, Risk Management"
3,O’dara Exotic Skincare,"Beauty, Flowers, Gift, Retail"
4,!Bewust sociaal op web,"Content Creators, Lead Generation, Sales, Soci..."
5,!Creatice,Education
6,!DOEVE*,"Advertising, Consulting, Marketing, Sales"
7,!FEST,Food and Beverage
8,!K7,"Media and Entertainment, Music"
9,!SHOUTTAG,"Marketing, Social Media Marketing"
10,!Woon,"Advice, Energy, Legal, Property Management"
11,!hey software,"CRM, Customer Service, E-Commerce, Information..."



LOCATION EXAMPLES


,name,locations
0,GivingTech,"Europe, Middle East, and Africa (EMEA), Middle..."
2,F. Rego,"European Union (EU), Europe, Middle East, and ..."
3,O’dara Exotic Skincare,"Greater Los Angeles Area, West Coast, Western US"
4,!Bewust sociaal op web,"European Union (EU), Europe, Middle East, and ..."
5,!Creatice,Asia-Pacific (APAC)
6,!DOEVE*,"European Union (EU), Europe, Middle East, and ..."
7,!FEST,"Europe, Middle East, and Africa (EMEA)"
8,!K7,"European Union (EU), Europe, Middle East, and ..."
9,!SHOUTTAG,"Greater Los Angeles Area, Inland Empire, West ..."
10,!Woon,"European Union (EU), Europe, Middle East, and ..."


In [27]:
# CELL 24 - SAMPLE UNIQUE CATEGORY / LOCATION STRINGS

sample_category_values = []
sample_location_values = []


for chunk in pd.read_csv(
    FEATURE_DATASET_FILE,
    usecols=[
        "categories",
        "locations"
    ],
    chunksize=100_000,
    low_memory=False
):

    if len(sample_category_values) < 200:

        category_values = (
            chunk["categories"]
            .dropna()
            .astype(str)
            .drop_duplicates()
            .head(
                200
                - len(sample_category_values)
            )
            .tolist()
        )

        sample_category_values.extend(
            category_values
        )


    if len(sample_location_values) < 200:

        location_values = (
            chunk["locations"]
            .dropna()
            .astype(str)
            .drop_duplicates()
            .head(
                200
                - len(sample_location_values)
            )
            .tolist()
        )

        sample_location_values.extend(
            location_values
        )


    if (
        len(sample_category_values) >= 200
        and len(sample_location_values) >= 200
    ):
        break


print(
    "Sample unique category strings:",
    len(sample_category_values)
)

print(
    "Sample unique location strings:",
    len(sample_location_values)
)


print("\nCATEGORY RAW VALUES")

for value in sample_category_values[:50]:
    print(repr(value))


print("\nLOCATION RAW VALUES")

for value in sample_location_values[:50]:
    print(repr(value))

Sample unique category strings: 200
Sample unique location strings: 200

CATEGORY RAW VALUES
'Financial Services, Insurance, Risk Management'
'Beauty, Flowers, Gift, Retail'
'Content Creators, Lead Generation, Sales, Social Media, Web Design'
'Education'
'Advertising, Consulting, Marketing, Sales'
'Food and Beverage'
'Media and Entertainment, Music'
'Marketing, Social Media Marketing'
'Advice, Energy, Legal, Property Management'
'CRM, Customer Service, E-Commerce, Information Technology, Internet, Messaging, Sales, Software, Telecommunications'
'Architecture, Interior Design, Landscaping'
'Apps, Education, Software, Video Games'
'Furniture, Manufacturing, Rental'
'Graphic Design, Printing, Retail'
'Consulting, Human Resources, Training'
'Accounting, Financial Services, Professional Services'
'Health Care'
'Consulting, Information Technology, Software'
'Information Technology, Network Security, Sales'
'Communities, Employment, Non Profit, Social'
'Logistics, Shipping, Transportation'
'L

In [28]:
# CELL 25 - AUDIT RETRIEVAL CORE FIELD AVAILABILITY

retrieval_core_stats = {
    "name_available": 0,
    "description_available": 0,
    "categories_available": 0,
    "locations_available": 0,
    "website_available": 0,
}

retrieval_total_rows = 0


for chunk in pd.read_csv(
    FEATURE_DATASET_FILE,
    usecols=[
        "name",
        "short_description",
        "categories",
        "locations",
        "website"
    ],
    chunksize=100_000,
    low_memory=False
):

    retrieval_total_rows += len(chunk)

    retrieval_core_stats[
        "name_available"
    ] += (
        chunk["name"]
        .notna()
        .sum()
    )

    retrieval_core_stats[
        "description_available"
    ] += (
        chunk["short_description"]
        .notna()
        .sum()
    )

    retrieval_core_stats[
        "categories_available"
    ] += (
        chunk["categories"]
        .notna()
        .sum()
    )

    retrieval_core_stats[
        "locations_available"
    ] += (
        chunk["locations"]
        .notna()
        .sum()
    )

    retrieval_core_stats[
        "website_available"
    ] += (
        chunk["website"]
        .notna()
        .sum()
    )


retrieval_core_availability_df = pd.DataFrame(
    [
        {
            "field": field,
            "available_count": count,
            "available_pct": (
                count
                / retrieval_total_rows
                * 100
            )
        }
        for field, count
        in retrieval_core_stats.items()
    ]
)


display(
    retrieval_core_availability_df
    .sort_values(
        "available_pct",
        ascending=False
    )
)

,field,available_count,available_pct
0,name_available,2807483,99.999751
1,description_available,2807440,99.998219
2,categories_available,2679953,95.457259
4,website_available,2657253,94.648708
3,locations_available,2624298,93.474883


In [34]:
# CELL 26 - DEFINE CLEAN RETRIEVAL TEXT BUILDER

employee_range_text_mapping = {
    "c_00001_00010": "1-10 employees",
    "c_00011_00050": "11-50 employees",
    "c_00051_00100": "51-100 employees",
    "c_00101_00250": "101-250 employees",
    "c_00251_00500": "251-500 employees",
    "c_00501_01000": "501-1000 employees",
    "c_01001_05000": "1001-5000 employees",
    "c_05001_10000": "5001-10000 employees",
    "c_10001_max": "10001+ employees",
}


def clean_year_text(series):

    numeric_year = pd.to_numeric(
        series,
        errors="coerce"
    )

    return (
        numeric_year
        .round()
        .astype("Int64")
        .astype("string")
        .fillna("")
    )


def build_retrieval_text(df):

    df = df.copy()


    employee_range_text = (
        df["num_employees_enum"]
        .map(
            employee_range_text_mapping
        )
        .fillna("")
    )


    founded_year_text = (
        clean_year_text(
            df["founded_year"]
        )
    )


    df["retrieval_text"] = (
        "Company: "
        + df["name"].fillna("").astype(str)

        + "\nDescription: "
        + df["short_description"].fillna("").astype(str)

        + "\nCategories: "
        + df["categories"].fillna("").astype(str)

        + "\nLocation: "
        + df["locations"].fillna("").astype(str)

        + "\nStatus: "
        + df["operating_status"].fillna("").astype(str)

        + "\nCompany Type: "
        + df["company_type"].fillna("").astype(str)

        + "\nIPO Status: "
        + df["ipo_status"].fillna("").astype(str)

        + "\nEmployee Range: "
        + employee_range_text.astype(str)

        + "\nFounded Year: "
        + founded_year_text.astype(str)

        + "\nLast Funding Type: "
        + df["last_funding_type"].fillna("").astype(str)
    )


    df["retrieval_text"] = (
        df["retrieval_text"]
        .str.replace(
            r"[ \t]+",
            " ",
            regex=True
        )
        .str.replace(
            r"\n+",
            "\n",
            regex=True
        )
        .str.strip()
    )


    return df

In [35]:
# CELL 27 - TEST RETRIEVAL TEXT ON SAMPLE

retrieval_sample_df = (
    build_retrieval_text(
        feature_sample_df.copy()
    )
)


print(
    "Sample retrieval rows:",
    len(retrieval_sample_df)
)


for i in range(
    min(
        10,
        len(retrieval_sample_df)
    )
):

    print("\n" + "=" * 100)

    print(
        retrieval_sample_df.iloc[i][
            "retrieval_text"
        ]
    )

Sample retrieval rows: 100

Company: GivingTech
Description: GivingTech - Giving Technologies, Inc is creating and leading a fundraising platform.
Categories: 
Location: Europe, Middle East, and Africa (EMEA), Middle East, Middle East and North Africa (MENA)
Status: active
Company Type: for_profit
IPO Status: private
Employee Range: 1-10 employees
Founded Year: 2020
Last Funding Type:

Company: X+
Description: X+ is an elite club for DeGods whales and builders who want to shape web3 and other future technologies.
Categories: 
Location: 
Status: active
Company Type: 
IPO Status: 
Employee Range: 
Founded Year: 2022
Last Funding Type:

Company: F. Rego
Description: F. Rego offers insurance consulting, brokerage, claim management, and risk management services.
Categories: Financial Services, Insurance, Risk Management
Location: European Union (EU), Europe, Middle East, and Africa (EMEA)
Status: active
Company Type: for_profit
IPO Status: private
Employee Range: 51-100 employees
Founded Ye

In [36]:
# CELL 28 - AUDIT RETRIEVAL TEXT LENGTH

retrieval_sample_df[
    "retrieval_text_length"
] = (
    retrieval_sample_df[
        "retrieval_text"
    ]
    .str.len()
)


retrieval_sample_df[
    "retrieval_word_count"
] = (
    retrieval_sample_df[
        "retrieval_text"
    ]
    .str.split()
    .str.len()
)


print("RETRIEVAL TEXT LENGTH SUMMARY")

print(
    retrieval_sample_df[
        [
            "retrieval_text_length",
            "retrieval_word_count"
        ]
    ].describe()
)


display(
    retrieval_sample_df[
        [
            "name",
            "retrieval_text_length",
            "retrieval_word_count",
            "retrieval_text"
        ]
    ]
    .sort_values(
        "retrieval_text_length",
        ascending=False
    )
    .head(10)
)

RETRIEVAL TEXT LENGTH SUMMARY
       retrieval_text_length  retrieval_word_count
count             100.000000             100.00000
mean              356.560000              47.08000
std                61.493421               7.99328
min               197.000000              24.00000
25%               321.500000              43.00000
50%               361.000000              48.00000
75%               394.500000              52.00000
max               529.000000              68.00000


,name,retrieval_text_length,retrieval_word_count,retrieval_text
50,#://CNXT,529,68,Company: #://CNXT\nDescription: A NeTWoRK MaNa...
48,#31 IT Solutions,493,66,Company: #31 IT Solutions\nDescription: #31 IT...
41,#1 Mental Health Boutique “Smart Meditation”,483,64,Company: #1 Mental Health Boutique “Smart Medi...
11,!hey software,465,57,Company: !hey software\nDescription: !hey soft...
55,#Die.Digitalfabrik,448,55,Company: #Die.Digitalfabrik\nDescription: #Die...
68,#LaPiscine,444,58,Company: #LaPiscine\nDescription: #LaP Piscine...
4,!Bewust sociaal op web,443,61,Company: !Bewust sociaal op web\nDescription: ...
18,"""Count On Us"" Income Tax & Bookkeeping",442,61,"Company: ""Count On Us"" Income Tax & Bookkeepin..."
76,#SocialSchool4EDU,441,54,Company: #SocialSchool4EDU\nDescription: #Soci...
58,#Evolve,428,49,Company: #Evolve\nDescription: #Evolve offers ...


In [37]:
# CELL 29 - BUILD FULL RETRIEVAL-READY DATASET

RETRIEVAL_READY_FILE = Path(
    "data/company_dataset/processed/company_dataset_retrieval_ready.csv"
)

RETRIEVAL_CHUNK_SIZE = 100_000


if RETRIEVAL_READY_FILE.exists():
    RETRIEVAL_READY_FILE.unlink()


rows_written = 0
first_write = True


for chunk_number, chunk in enumerate(
    pd.read_csv(
        FEATURE_DATASET_FILE,
        chunksize=RETRIEVAL_CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    retrieval_chunk = build_retrieval_text(
        chunk
    )


    retrieval_chunk.to_csv(
        RETRIEVAL_READY_FILE,
        mode="w" if first_write else "a",
        header=first_write,
        index=False
    )


    first_write = False

    rows_written += len(
        retrieval_chunk
    )


    print(
        f"Chunk {chunk_number} "
        f"| rows written: {rows_written:,}"
    )


print("\nRETRIEVAL DATASET COMPLETE")

print(
    "Rows written:",
    rows_written
)

print(
    "Output:",
    RETRIEVAL_READY_FILE
)

Chunk 1 | rows written: 100,000
Chunk 2 | rows written: 200,000
Chunk 3 | rows written: 300,000
Chunk 4 | rows written: 400,000
Chunk 5 | rows written: 500,000
Chunk 6 | rows written: 600,000
Chunk 7 | rows written: 700,000
Chunk 8 | rows written: 800,000
Chunk 9 | rows written: 900,000
Chunk 10 | rows written: 1,000,000
Chunk 11 | rows written: 1,100,000
Chunk 12 | rows written: 1,200,000
Chunk 13 | rows written: 1,300,000
Chunk 14 | rows written: 1,400,000
Chunk 15 | rows written: 1,500,000
Chunk 16 | rows written: 1,600,000
Chunk 17 | rows written: 1,700,000
Chunk 18 | rows written: 1,800,000
Chunk 19 | rows written: 1,900,000
Chunk 20 | rows written: 2,000,000
Chunk 21 | rows written: 2,100,000
Chunk 22 | rows written: 2,200,000
Chunk 23 | rows written: 2,300,000
Chunk 24 | rows written: 2,400,000
Chunk 25 | rows written: 2,500,000
Chunk 26 | rows written: 2,600,000
Chunk 27 | rows written: 2,700,000
Chunk 28 | rows written: 2,800,000
Chunk 29 | rows written: 2,807,490

RETRIEVAL D

In [38]:
# CELL 30 - AUDIT FINAL RETRIEVAL DATASET

retrieval_total_rows = 0
retrieval_empty_count = 0
retrieval_length_sum = 0
retrieval_word_sum = 0
retrieval_min_length = None
retrieval_max_length = 0


for chunk in pd.read_csv(
    RETRIEVAL_READY_FILE,
    usecols=[
        "id",
        "retrieval_text"
    ],
    chunksize=RETRIEVAL_CHUNK_SIZE,
    low_memory=False
):

    retrieval_total_rows += len(chunk)


    text_series = (
        chunk[
            "retrieval_text"
        ]
        .fillna("")
        .astype(str)
    )


    text_lengths = (
        text_series
        .str.len()
    )


    word_counts = (
        text_series
        .str.split()
        .str.len()
    )


    retrieval_empty_count += (
        text_series
        .str.strip()
        .eq("")
        .sum()
    )


    retrieval_length_sum += (
        text_lengths.sum()
    )


    retrieval_word_sum += (
        word_counts.sum()
    )


    chunk_min = (
        text_lengths.min()
    )

    chunk_max = (
        text_lengths.max()
    )


    if retrieval_min_length is None:
        retrieval_min_length = chunk_min
    else:
        retrieval_min_length = min(
            retrieval_min_length,
            chunk_min
        )


    retrieval_max_length = max(
        retrieval_max_length,
        chunk_max
    )


print("FINAL RETRIEVAL AUDIT")

print(
    "Rows:",
    retrieval_total_rows
)

print(
    "Empty retrieval text:",
    retrieval_empty_count
)

print(
    "Average characters:",
    round(
        retrieval_length_sum
        / retrieval_total_rows,
        2
    )
)

print(
    "Average words:",
    round(
        retrieval_word_sum
        / retrieval_total_rows,
        2
    )
)

print(
    "Minimum characters:",
    retrieval_min_length
)

print(
    "Maximum characters:",
    retrieval_max_length
)

FINAL RETRIEVAL AUDIT
Rows: 2807490
Empty retrieval text: 0
Average characters: 369.76
Average words: 47.91
Minimum characters: 145
Maximum characters: 1918


In [39]:
# CELL 31 - AUDIT BASIC CANDIDATE UNIVERSE

candidate_stats = {
    "total_rows": 0,
    "active_rows": 0,
    "closed_rows": 0,
    "with_categories": 0,
    "with_locations": 0,
    "with_employee_data": 0,
    "with_founded_year": 0,
}


for chunk in pd.read_csv(
    RETRIEVAL_READY_FILE,
    usecols=[
        "operating_status",
        "categories",
        "locations",
        "employee_min",
        "founded_year"
    ],
    chunksize=RETRIEVAL_CHUNK_SIZE,
    low_memory=False
):

    candidate_stats[
        "total_rows"
    ] += len(chunk)


    candidate_stats[
        "active_rows"
    ] += (
        chunk[
            "operating_status"
        ]
        .eq("active")
        .sum()
    )


    candidate_stats[
        "closed_rows"
    ] += (
        chunk[
            "operating_status"
        ]
        .eq("closed")
        .sum()
    )


    candidate_stats[
        "with_categories"
    ] += (
        chunk[
            "categories"
        ]
        .notna()
        .sum()
    )


    candidate_stats[
        "with_locations"
    ] += (
        chunk[
            "locations"
        ]
        .notna()
        .sum()
    )


    candidate_stats[
        "with_employee_data"
    ] += (
        chunk[
            "employee_min"
        ]
        .notna()
        .sum()
    )


    candidate_stats[
        "with_founded_year"
    ] += (
        chunk[
            "founded_year"
        ]
        .notna()
        .sum()
    )


candidate_audit_df = pd.DataFrame(
    [
        {
            "metric": key,
            "count": value,
            "pct_of_total": (
                value
                / candidate_stats["total_rows"]
                * 100
                if key != "total_rows"
                else 100
            )
        }
        for key, value
        in candidate_stats.items()
    ]
)


display(
    candidate_audit_df
)

,metric,count,pct_of_total
0,total_rows,2807490,100.000000
1,active_rows,2671042,95.139858
2,closed_rows,136448,4.860142
3,with_categories,2679953,95.457259
4,with_locations,2624298,93.474883
5,with_employee_data,2370227,84.425127
6,with_founded_year,2322078,82.710108


In [40]:
# CELL 32 - AUDIT EMPLOYEE SIZE DISTRIBUTION

employee_size_counts = {}


for chunk in pd.read_csv(
    RETRIEVAL_READY_FILE,
    usecols=[
        "num_employees_enum"
    ],
    chunksize=RETRIEVAL_CHUNK_SIZE,
    low_memory=False
):

    counts = (
        chunk[
            "num_employees_enum"
        ]
        .fillna("<MISSING>")
        .value_counts()
    )


    for key, value in counts.items():

        employee_size_counts[
            key
        ] = (
            employee_size_counts.get(
                key,
                0
            )
            + value
        )


employee_size_distribution_df = (
    pd.DataFrame(
        [
            {
                "employee_range":
                    key,

                "count":
                    value,

                "pct_of_total":
                    (
                        value
                        / candidate_stats[
                            "total_rows"
                        ]
                        * 100
                    )
            }
            for key, value
            in employee_size_counts.items()
        ]
    )
    .sort_values(
        "count",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    employee_size_distribution_df
)

,employee_range,count,pct_of_total
0,c_00011_00050,996943,35.510118
1,c_00001_00010,779854,27.777623
2,<MISSING>,437263,15.574873
3,c_00051_00100,227712,8.110875
4,c_00101_00250,173680,6.186309
5,c_00251_00500,76536,2.726136
6,c_00501_01000,50480,1.798047
7,c_01001_05000,43534,1.550638
8,c_10001_max,11475,0.408728
9,c_05001_10000,10013,0.356653


In [41]:
# CELL 33 - AUDIT TOP COMPANY CATEGORIES

category_counts = {}


for chunk_number, chunk in enumerate(
    pd.read_csv(
        RETRIEVAL_READY_FILE,
        usecols=[
            "categories"
        ],
        chunksize=RETRIEVAL_CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    category_series = (
        chunk[
            "categories"
        ]
        .dropna()
        .astype(str)
        .str.split(",")
        .explode()
        .str.strip()
    )


    category_series = (
        category_series[
            category_series.ne("")
        ]
    )


    counts = (
        category_series
        .value_counts()
    )


    for category, count in counts.items():

        category_counts[
            category
        ] = (
            category_counts.get(
                category,
                0
            )
            + count
        )


    print(
        f"Processed category chunk "
        f"{chunk_number}"
    )


category_frequency_df = (
    pd.DataFrame(
        [
            {
                "category":
                    category,

                "company_count":
                    count,

                "pct_of_companies":
                    (
                        count
                        / candidate_stats[
                            "total_rows"
                        ]
                        * 100
                    )
            }
            for category, count
            in category_counts.items()
        ]
    )
    .sort_values(
        "company_count",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "Unique categories:",
    len(
        category_frequency_df
    )
)


display(
    category_frequency_df
    .head(100)
)

Processed category chunk 1
Processed category chunk 2
Processed category chunk 3
Processed category chunk 4
Processed category chunk 5
Processed category chunk 6
Processed category chunk 7
Processed category chunk 8
Processed category chunk 9
Processed category chunk 10
Processed category chunk 11
Processed category chunk 12
Processed category chunk 13
Processed category chunk 14
Processed category chunk 15
Processed category chunk 16
Processed category chunk 17
Processed category chunk 18
Processed category chunk 19
Processed category chunk 20
Processed category chunk 21
Processed category chunk 22
Processed category chunk 23
Processed category chunk 24
Processed category chunk 25
Processed category chunk 26
Processed category chunk 27
Processed category chunk 28
Processed category chunk 29
Unique categories: 802


,category,company_count,pct_of_companies
0,Manufacturing,321095,11.437084
1,Software,290197,10.336528
2,Consulting,275717,9.820765
3,Information Technology,272569,9.708637
4,Health Care,237855,8.472158
...,...,...,...
95,Venture Capital,21390,0.761891
96,Fitness,20511,0.730581
97,Oil and Gas,20441,0.728088
98,Agriculture,20220,0.720216


In [42]:
# CELL 34 - DEFINE CANDIDATE FILTER FUNCTION

def filter_company_candidates(
    df,
    *,
    active_only=True,
    categories=None,
    location_keywords=None,
    employee_min=None,
    employee_max=None,
    founded_year_min=None,
    founded_year_max=None,
):
    """
    Deterministic pre-filter for company retrieval.

    Parameters
    ----------
    df : pandas.DataFrame

    active_only : bool
        Keep only operating_status == "active".

    categories : list[str] | None
        Match if ANY requested category is present.

    location_keywords : list[str] | None
        Match if ANY keyword occurs in locations text.

    employee_min : int | None
        Requested minimum company size.

    employee_max : int | None
        Requested maximum company size.

    founded_year_min : int | None

    founded_year_max : int | None
    """

    mask = pd.Series(
        True,
        index=df.index
    )


    # ----------------------------------
    # Active status
    # ----------------------------------

    if active_only:

        mask &= (
            df["operating_status"]
            .eq("active")
        )


    # ----------------------------------
    # Category filter
    # ----------------------------------

    if categories:

        requested_categories = {
            str(category)
            .strip()
            .lower()

            for category in categories

            if str(category).strip()
        }


        category_match = pd.Series(
            False,
            index=df.index
        )


        category_text = (
            df["categories"]
            .fillna("")
            .astype(str)
            .str.lower()
        )


        for category in requested_categories:

            category_match |= (
                category_text
                .str.contains(
                    category,
                    regex=False
                )
            )


        mask &= category_match


    # ----------------------------------
    # Location filter
    # ----------------------------------

    if location_keywords:

        location_match = pd.Series(
            False,
            index=df.index
        )


        location_text = (
            df["locations"]
            .fillna("")
            .astype(str)
            .str.lower()
        )


        for keyword in location_keywords:

            keyword = (
                str(keyword)
                .strip()
                .lower()
            )

            if keyword:

                location_match |= (
                    location_text
                    .str.contains(
                        keyword,
                        regex=False
                    )
                )


        mask &= location_match


    # ----------------------------------
    # Employee range overlap
    # ----------------------------------

    if (
        employee_min is not None
        or employee_max is not None
    ):

        company_min = pd.to_numeric(
            df["employee_min"],
            errors="coerce"
        )

        company_max = pd.to_numeric(
            df["employee_max"],
            errors="coerce"
        )


        if employee_min is not None:

            # Open-ended company_max is allowed
            mask &= (
                company_max.isna()
                |
                (
                    company_max
                    >= employee_min
                )
            )


        if employee_max is not None:

            mask &= (
                company_min.isna()
                |
                (
                    company_min
                    <= employee_max
                )
            )


    # ----------------------------------
    # Founded year
    # ----------------------------------

    founded_year = pd.to_numeric(
        df["founded_year"],
        errors="coerce"
    )


    if founded_year_min is not None:

        mask &= (
            founded_year
            >= founded_year_min
        )


    if founded_year_max is not None:

        mask &= (
            founded_year
            <= founded_year_max
        )


    return (
        df.loc[mask]
        .copy()
    )

In [43]:
# CELL 35 - TEST CANDIDATE FILTER ON SMALL SUBSET

candidate_test_df = pd.read_csv(
    RETRIEVAL_READY_FILE,
    nrows=200_000,
    low_memory=False
)


test_candidates = filter_company_candidates(
    candidate_test_df,

    active_only=True,

    categories=[
        "Software",
        "Information Technology",
    ],

    employee_min=1,
    employee_max=250,
)


print(
    "Input rows:",
    len(candidate_test_df)
)

print(
    "Candidate rows:",
    len(test_candidates)
)

print(
    "Reduction:",
    round(
        (
            1
            -
            len(test_candidates)
            / len(candidate_test_df)
        )
        * 100,
        2
    ),
    "%"
)


display(
    test_candidates[
        [
            "id",
            "name",
            "categories",
            "locations",
            "num_employees_enum",
            "founded_year"
        ]
    ]
    .head(30)
)

Input rows: 200000
Candidate rows: 31200
Reduction: 84.4 %


,id,name,categories,locations,num_employees_enum,founded_year
13,7ef2d009-800d-c9b6-0f15-5f52c1c6c3e1,!mpossible,"Apps, Education, Software, Video Games","European Union (EU), Europe, Middle East, and ...",c_00001_00010,NaN
21,9a88ec8e-ec23-4050-9bb7-d7b3ec7252f8,"""Establish"" d.o.o. Sarajevo","Consulting, Information Technology, Software","Europe, Middle East, and Africa (EMEA)",c_00011_00050,2000.0
22,ec3f207d-0a73-4a3d-a0d0-6a8b3d33b2eb,"""First"" Sales & Marketing","Information Technology, Network Security, Sales","European Union (EU), Europe, Middle East, and ...",c_00001_00010,NaN
27,5f8a5198-53e5-4282-945f-fa5f070801f7,"""Ocadeus"" Monitoring System","Developer Tools, Information Technology, Inter...",NaN,c_00011_00050,2017.0
32,bf314f39-32ff-4415-800e-b13fc6424fa2,"""Text & Technik"" - Webseiten-Optimierung","Analytics, Information Technology, Internet","European Union (EU), Europe, Middle East, and ...",c_00001_00010,NaN
33,a90ed59c-8be2-43e4-9e85-884af9493569,"""VTI"" PrJSC","Computer, Software","Europe, Middle East, and Africa (EMEA)",c_00101_00250,1972.0
41,e3b47971-dbe8-49c3-85be-4f203649cbdf,#1 Mental Health Boutique “Smart Meditation”,"Health Care, Information Technology, Mental He...","Europe, Middle East, and Africa (EMEA), Gulf C...",c_00011_00050,2022.0
46,4178a33b-f674-a48c-72cd-24b9d69d7f2f,#10,"Consulting, E-Commerce, Shopping, Software","San Francisco Bay Area, West Coast, Western US",c_00011_00050,2000.0
47,0ae10768-f06b-4b78-a802-96c619a4cca5,#2,Information and Communications Technology (ICT...,"European Union (EU), Europe, Middle East, and ...",c_00001_00010,2000.0
48,f9a1d3c5-cc6d-49fa-9ef5-04e342ce54e1,#31 IT Solutions,"Digital Marketing, Graphic Design, Software, W...","Europe, Middle East, and Africa (EMEA), Gulf C...",c_00011_00050,2018.0


In [44]:
# CELL 36 - TEST MULTIPLE PREFILTER SCENARIOS

test_scenarios = [
    {
        "scenario": "Software / IT",
        "kwargs": {
            "categories": [
                "Software",
                "Information Technology",
            ]
        }
    },

    {
        "scenario": "Small Software / IT",
        "kwargs": {
            "categories": [
                "Software",
                "Information Technology",
            ],
            "employee_min": 1,
            "employee_max": 50,
        }
    },

    {
        "scenario": "Software / IT founded 2015+",
        "kwargs": {
            "categories": [
                "Software",
                "Information Technology",
            ],
            "founded_year_min": 2015,
        }
    },

    {
        "scenario": "Small Software / IT founded 2015+",
        "kwargs": {
            "categories": [
                "Software",
                "Information Technology",
            ],
            "employee_min": 1,
            "employee_max": 50,
            "founded_year_min": 2015,
        }
    },
]


scenario_results = []


for scenario in test_scenarios:

    result = filter_company_candidates(
        candidate_test_df,
        active_only=True,
        **scenario["kwargs"]
    )


    scenario_results.append(
        {
            "scenario":
                scenario["scenario"],

            "candidate_count":
                len(result),

            "pct_of_sample":
                (
                    len(result)
                    / len(candidate_test_df)
                    * 100
                ),

            "reduction_pct":
                (
                    1
                    -
                    len(result)
                    / len(candidate_test_df)
                )
                * 100
        }
    )


scenario_results_df = pd.DataFrame(
    scenario_results
)


display(
    scenario_results_df
)

,scenario,candidate_count,pct_of_sample,reduction_pct
0,Software / IT,32798,16.3990,83.6010
1,Small Software / IT,26668,13.3340,86.6660
2,Software / IT founded 2015+,8849,4.4245,95.5755
3,Small Software / IT founded 2015+,7887,3.9435,96.0565


In [45]:
# CELL 37 - DEFINE EXACT CATEGORY MATCH HELPER

def exact_category_mask(
    series,
    requested_categories
):
    """
    Match exact comma-separated categories.
    Case-insensitive.
    """

    requested = {
        str(category)
        .strip()
        .lower()

        for category in requested_categories

        if str(category).strip()
    }


    if not requested:

        return pd.Series(
            False,
            index=series.index
        )


    category_lists = (
        series
        .fillna("")
        .astype(str)
        .str.lower()
        .str.split(",")
    )


    return category_lists.apply(
        lambda values:
            any(
                value.strip() in requested
                for value in values
            )
    )

In [46]:
# CELL 38 - COMPARE SUBSTRING VS EXACT CATEGORY MATCH

sample_categories = (
    candidate_test_df[
        "categories"
    ]
)


substring_mask = (
    sample_categories
    .fillna("")
    .astype(str)
    .str.lower()
    .str.contains(
        "software",
        regex=False
    )
)


exact_mask = exact_category_mask(
    sample_categories,
    [
        "Software"
    ]
)


comparison_df = pd.DataFrame(
    {
        "substring_match":
            substring_mask,

        "exact_match":
            exact_mask
    }
)


print(
    "Substring matches:",
    substring_mask.sum()
)

print(
    "Exact matches:",
    exact_mask.sum()
)

print(
    "Substring-only matches:",
    (
        substring_mask
        &
        ~exact_mask
    ).sum()
)


substring_only_examples = (
    candidate_test_df.loc[
        substring_mask
        &
        ~exact_mask,
        [
            "name",
            "categories"
        ]
    ]
    .head(30)
)


display(
    substring_only_examples
)

Substring matches: 22076
Exact matches: 21186
Substring-only matches: 890


,name,categories
82,#VA,"SEO, Software Engineering, Web Development"
168,&do,"Consulting, Digital Marketing, Product Design,..."
282,*instinctools,"B2B, Big Data, Blockchain, Business Intelligen..."
408,.expositio,Software Engineering
421,.seed,"Career Planning, Information Technology, Inter..."
435,/community,"Enterprise Software, Messaging, SaaS, Web Deve..."
485,01 Enterprise Limited,"Advertising, E-Commerce, Email Marketing, Ente..."
722,0xcompany,"Information Technology, Open Source, Software ..."
894,1 Second Everyday,"Computer, Enterprise Software, Information Tec..."
1054,1-800-GOT-JUNK?,"Customer Service, Enterprise Software, Recycling"


In [47]:
# CELL 39 - FULL UNIVERSE PREFILTER AUDIT

full_scenario_counts = {
    "Software / IT": 0,
    "Small Software / IT": 0,
    "Software / IT founded 2015+": 0,
    "Small Software / IT founded 2015+": 0,
}

full_total_rows = 0


requested_categories = {
    "software",
    "information technology"
}


for chunk_number, chunk in enumerate(
    pd.read_csv(
        RETRIEVAL_READY_FILE,
        usecols=[
            "operating_status",
            "categories",
            "employee_min",
            "employee_max",
            "founded_year"
        ],
        chunksize=RETRIEVAL_CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    full_total_rows += len(chunk)


    # --------------------------------
    # Active
    # --------------------------------

    active_mask = (
        chunk[
            "operating_status"
        ]
        .eq("active")
    )


    # --------------------------------
    # Exact category matching
    # vectorized with split + explode
    # --------------------------------

    category_df = (
        chunk[
            ["categories"]
        ]
        .copy()
    )


    exploded = (
        category_df[
            "categories"
        ]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.split(",")
        .explode()
        .str.strip()
    )


    matching_indexes = (
        exploded[
            exploded.isin(
                requested_categories
            )
        ]
        .index
        .unique()
    )


    category_mask = (
        chunk.index.isin(
            matching_indexes
        )
    )


    # --------------------------------
    # Employee size overlap: 1-50
    # --------------------------------

    employee_min_series = pd.to_numeric(
        chunk[
            "employee_min"
        ],
        errors="coerce"
    )


    employee_max_series = pd.to_numeric(
        chunk[
            "employee_max"
        ],
        errors="coerce"
    )


    small_company_mask = (
        (
            employee_max_series.isna()
            |
            (
                employee_max_series
                >= 1
            )
        )
        &
        (
            employee_min_series.isna()
            |
            (
                employee_min_series
                <= 50
            )
        )
    )


    # --------------------------------
    # Founded 2015+
    # --------------------------------

    founded_year_series = pd.to_numeric(
        chunk[
            "founded_year"
        ],
        errors="coerce"
    )


    founded_2015_mask = (
        founded_year_series
        >= 2015
    )


    # --------------------------------
    # Scenario masks
    # --------------------------------

    software_it_mask = (
        active_mask
        &
        category_mask
    )


    small_software_it_mask = (
        software_it_mask
        &
        small_company_mask
    )


    software_it_2015_mask = (
        software_it_mask
        &
        founded_2015_mask
    )


    small_software_it_2015_mask = (
        software_it_mask
        &
        small_company_mask
        &
        founded_2015_mask
    )


    full_scenario_counts[
        "Software / IT"
    ] += software_it_mask.sum()


    full_scenario_counts[
        "Small Software / IT"
    ] += small_software_it_mask.sum()


    full_scenario_counts[
        "Software / IT founded 2015+"
    ] += software_it_2015_mask.sum()


    full_scenario_counts[
        "Small Software / IT founded 2015+"
    ] += small_software_it_2015_mask.sum()


    print(
        f"Processed full-universe chunk "
        f"{chunk_number}"
    )


full_scenario_results_df = pd.DataFrame(
    [
        {
            "scenario":
                scenario,

            "candidate_count":
                count,

            "pct_of_total":
                (
                    count
                    / full_total_rows
                    * 100
                ),

            "reduction_pct":
                (
                    1
                    -
                    count
                    / full_total_rows
                )
                * 100
        }
        for scenario, count
        in full_scenario_counts.items()
    ]
)


display(
    full_scenario_results_df
)

Processed full-universe chunk 1
Processed full-universe chunk 2
Processed full-universe chunk 3
Processed full-universe chunk 4
Processed full-universe chunk 5
Processed full-universe chunk 6
Processed full-universe chunk 7
Processed full-universe chunk 8
Processed full-universe chunk 9
Processed full-universe chunk 10
Processed full-universe chunk 11
Processed full-universe chunk 12
Processed full-universe chunk 13
Processed full-universe chunk 14
Processed full-universe chunk 15
Processed full-universe chunk 16
Processed full-universe chunk 17
Processed full-universe chunk 18
Processed full-universe chunk 19
Processed full-universe chunk 20
Processed full-universe chunk 21
Processed full-universe chunk 22
Processed full-universe chunk 23
Processed full-universe chunk 24
Processed full-universe chunk 25
Processed full-universe chunk 26
Processed full-universe chunk 27
Processed full-universe chunk 28
Processed full-universe chunk 29


,scenario,candidate_count,pct_of_total,reduction_pct
0,Software / IT,415602,14.803330,85.196670
1,Small Software / IT,342265,12.191139,87.808861
2,Software / IT founded 2015+,129334,4.606748,95.393252
3,Small Software / IT founded 2015+,115798,4.124610,95.875390


In [48]:
# CELL 40 - DEFINE FULL DATASET CANDIDATE EXTRACTOR

def extract_candidates_from_file(
    input_file,
    *,
    categories=None,
    active_only=True,
    employee_min=None,
    employee_max=None,
    founded_year_min=None,
    founded_year_max=None,
    location_keywords=None,
    chunksize=100_000,
):
    """
    Stream the large retrieval dataset and return only matching candidates.
    """

    candidate_chunks = []

    requested_categories = None

    if categories:
        requested_categories = {
            str(category).strip().lower()
            for category in categories
            if str(category).strip()
        }


    for chunk in pd.read_csv(
        input_file,
        chunksize=chunksize,
        low_memory=False
    ):

        mask = pd.Series(
            True,
            index=chunk.index
        )


        # -------------------------
        # Operating status
        # -------------------------

        if active_only:

            mask &= (
                chunk["operating_status"]
                .eq("active")
            )


        # -------------------------
        # Exact category matching
        # -------------------------

        if requested_categories:

            exploded = (
                chunk["categories"]
                .fillna("")
                .astype(str)
                .str.lower()
                .str.split(",")
                .explode()
                .str.strip()
            )

            matching_indexes = (
                exploded[
                    exploded.isin(
                        requested_categories
                    )
                ]
                .index
                .unique()
            )

            mask &= (
                chunk.index.isin(
                    matching_indexes
                )
            )


        # -------------------------
        # Employee range overlap
        # -------------------------

        company_min = pd.to_numeric(
            chunk["employee_min"],
            errors="coerce"
        )

        company_max = pd.to_numeric(
            chunk["employee_max"],
            errors="coerce"
        )


        if employee_min is not None:

            mask &= (
                company_max.isna()
                |
                (
                    company_max
                    >= employee_min
                )
            )


        if employee_max is not None:

            mask &= (
                company_min.isna()
                |
                (
                    company_min
                    <= employee_max
                )
            )


        # -------------------------
        # Founded year
        # -------------------------

        founded_year = pd.to_numeric(
            chunk["founded_year"],
            errors="coerce"
        )


        if founded_year_min is not None:

            mask &= (
                founded_year
                >= founded_year_min
            )


        if founded_year_max is not None:

            mask &= (
                founded_year
                <= founded_year_max
            )


        # -------------------------
        # Location keyword matching
        # -------------------------

        if location_keywords:

            location_text = (
                chunk["locations"]
                .fillna("")
                .astype(str)
                .str.lower()
            )

            location_mask = pd.Series(
                False,
                index=chunk.index
            )

            for keyword in location_keywords:

                keyword = (
                    str(keyword)
                    .strip()
                    .lower()
                )

                if keyword:

                    location_mask |= (
                        location_text
                        .str.contains(
                            keyword,
                            regex=False
                        )
                    )

            mask &= location_mask


        matches = (
            chunk.loc[mask]
            .copy()
        )


        if len(matches) > 0:

            candidate_chunks.append(
                matches
            )


    if not candidate_chunks:

        return pd.DataFrame()


    return pd.concat(
        candidate_chunks,
        ignore_index=True
    )

In [49]:
# CELL 41 - EXTRACT REALISTIC SOFTWARE / IT CANDIDATES

software_it_candidates_df = (
    extract_candidates_from_file(
        RETRIEVAL_READY_FILE,

        categories=[
            "Software",
            "Information Technology",
        ],

        active_only=True,

        employee_min=1,
        employee_max=50,

        founded_year_min=2015,

        chunksize=RETRIEVAL_CHUNK_SIZE,
    )
)


print(
    "Candidates extracted:",
    len(
        software_it_candidates_df
    )
)


print(
    "Columns:",
    len(
        software_it_candidates_df.columns
    )
)


display(
    software_it_candidates_df[
        [
            "id",
            "name",
            "short_description",
            "categories",
            "locations",
            "num_employees_enum",
            "founded_year",
            "retrieval_text",
        ]
    ]
    .head(20)
)

Candidates extracted: 115798
Columns: 43


,id,name,short_description,categories,locations,num_employees_enum,founded_year,retrieval_text
0,5f8a5198-53e5-4282-945f-fa5f070801f7,"""Ocadeus"" Monitoring System","""Ocadeus"" is a monitoring system which will in...","Developer Tools, Information Technology, Inter...",NaN,c_00011_00050,2017.0,"Company: ""Ocadeus"" Monitoring System\nDescript..."
1,e3b47971-dbe8-49c3-85be-4f203649cbdf,#1 Mental Health Boutique “Smart Meditation”,"Mobile Application, Application for VR, SaaS, ...","Health Care, Information Technology, Mental He...","Europe, Middle East, and Africa (EMEA), Gulf C...",c_00011_00050,2022.0,Company: #1 Mental Health Boutique “Smart Medi...
2,f9a1d3c5-cc6d-49fa-9ef5-04e342ce54e1,#31 IT Solutions,#31 IT Solutions offers IT services and soluti...,"Digital Marketing, Graphic Design, Software, W...","Europe, Middle East, and Africa (EMEA), Gulf C...",c_00011_00050,2018.0,Company: #31 IT Solutions\nDescription: #31 IT...
3,fae3f75b-ce54-f05b-9d2d-c8b1ac8b368b,#://CNXT,A NeTWoRK MaNaGeMeNT LeDGeR of SySTeMs CNXTing...,"Consumer Software, Enterprise Software, Financ...","Greater New York Area, East Coast, Northeaster...",c_00011_00050,2017.0,Company: #://CNXT\nDescription: A NeTWoRK MaNa...
4,3f0471d3-1251-429f-a72b-087d4df42614,#Dovyo,"#Dovyo is a marketing, sales, and support cust...","Information Technology, Software",Asia-Pacific (APAC),c_00011_00050,2022.0,Company: #Dovyo\nDescription: #Dovyo is a mark...
5,7c9b7cd5-f87b-4f46-b3b4-d380e72881c5,#Stratapp,#Stratapp is a remote work app that helps with...,"Apps, Information Technology, IT Management","Asia-Pacific (APAC), Australasia",c_00011_00050,2017.0,Company: #Stratapp\nDescription: #Stratapp is ...
6,84286c33-a42d-4eb3-a8a4-7abfd64bae82,#team,#team is a task app for managing repeating tas...,"Apps, SaaS, Software","Asia-Pacific (APAC), Australasia",c_00011_00050,2016.0,Company: #team\nDescription: #team is a task a...
7,4ff3e834-54a1-4a9b-845b-b0991e306536,$://SWiPe,#://CNXT $://SWiPe x $://THeSTRiPeDesK x $://T...,"Billing, Finance, Financial Exchanges, Financi...","Greater Chicago Area, Great Lakes, Midwestern US",c_00011_00050,2021.0,Company: $://SWiPe\nDescription: #://CNXT $://...
8,8b89d048-8da4-4002-a4e0-8fdeaba79c6c,$://THeXDesK,THe B2B x B2G financial software LeDGeR of ReC...,"B2B, B2C, Big Data, Enterprise Software, Finan...","Greater Chicago Area, Great Lakes, Midwestern US",c_00011_00050,2021.0,Company: $://THeXDesK\nDescription: THe B2B x ...
9,dba094a8-224b-4656-8519-6bd27d05e196,&Charge,&Charge is the only European platform combinin...,"Advertising Platforms, Automotive, Big Data, G...","European Union (EU), Europe, Middle East, and ...",c_00011_00050,2019.0,Company: &Charge\nDescription: &Charge is the ...


In [50]:
# CELL 42 - AUDIT EXTRACTED CANDIDATE SET

candidate_text_lengths = (
    software_it_candidates_df[
        "retrieval_text"
    ]
    .fillna("")
    .astype(str)
    .str.len()
)


candidate_word_counts = (
    software_it_candidates_df[
        "retrieval_text"
    ]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)


print("CANDIDATE SET AUDIT")

print(
    "Rows:",
    len(
        software_it_candidates_df
    )
)

print(
    "Unique IDs:",
    software_it_candidates_df[
        "id"
    ].nunique()
)

print(
    "Average characters:",
    round(
        candidate_text_lengths.mean(),
        2
    )
)

print(
    "Average words:",
    round(
        candidate_word_counts.mean(),
        2
    )
)

print(
    "Max words:",
    candidate_word_counts.max()
)

print(
    "Missing retrieval text:",
    software_it_candidates_df[
        "retrieval_text"
    ]
    .isna()
    .sum()
)

CANDIDATE SET AUDIT
Rows: 115798
Unique IDs: 115798
Average characters: 382.22
Average words: 49.24
Max words: 242
Missing retrieval text: 0


In [53]:
# CELL 43 - LOAD EMBEDDING MODEL

from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_PATH = "models/multilingual-minilm"


embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_PATH
)


print(
    "Embedding model loaded:"
)

print(
    EMBEDDING_MODEL_PATH
)

e:\BINUS_CODING\Ai Builders Hackhaton\AI-Builders-Hackhaton-2026-Backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1806.15it/s]


Embedding model loaded:
models/multilingual-minilm


In [54]:
# CELL 44 - BENCHMARK EMBEDDING SPEED ON 1,000 CANDIDATES

import time


EMBEDDING_BENCHMARK_SIZE = 1_000


embedding_benchmark_df = (
    software_it_candidates_df[
        [
            "id",
            "name",
            "retrieval_text"
        ]
    ]
    .head(
        EMBEDDING_BENCHMARK_SIZE
    )
    .copy()
)


benchmark_texts = (
    embedding_benchmark_df[
        "retrieval_text"
    ]
    .fillna("")
    .astype(str)
    .tolist()
)


start_time = time.perf_counter()


benchmark_embeddings = (
    embedding_model.encode(
        benchmark_texts,

        batch_size=64,

        show_progress_bar=True,

        convert_to_numpy=True,

        normalize_embeddings=True,
    )
)


elapsed_seconds = (
    time.perf_counter()
    - start_time
)


rows_per_second = (
    EMBEDDING_BENCHMARK_SIZE
    / elapsed_seconds
)


estimated_full_seconds = (
    len(
        software_it_candidates_df
    )
    / rows_per_second
)


print("\nEMBEDDING BENCHMARK")

print(
    "Rows:",
    EMBEDDING_BENCHMARK_SIZE
)

print(
    "Embedding shape:",
    benchmark_embeddings.shape
)

print(
    "Elapsed seconds:",
    round(
        elapsed_seconds,
        2
    )
)

print(
    "Rows / second:",
    round(
        rows_per_second,
        2
    )
)

print(
    "Estimated time for "
    f"{len(software_it_candidates_df):,} rows:"
)

print(
    round(
        estimated_full_seconds / 60,
        2
    ),
    "minutes"
)

Batches: 100%|██████████| 16/16 [00:30<00:00,  1.94s/it]


EMBEDDING BENCHMARK
Rows: 1000
Embedding shape: (1000, 384)
Elapsed seconds: 31.02
Rows / second: 32.23
Estimated time for 115,798 rows:
59.88 minutes


In [55]:
# CELL 45 - TEST SEMANTIC RANKING ON BENCHMARK SUBSET

import numpy as np


test_query = """
A small software startup building an AI-powered SaaS
platform for businesses. The company is early-stage,
has a small team, and provides software and information
technology services.
""".strip()


query_embedding = (
    embedding_model.encode(
        [test_query],

        convert_to_numpy=True,

        normalize_embeddings=True,
    )[0]
)


similarity_scores = (
    benchmark_embeddings
    @ query_embedding
)


semantic_test_results_df = (
    embedding_benchmark_df
    .copy()
)


semantic_test_results_df[
    "semantic_score"
] = similarity_scores


semantic_test_results_df = (
    semantic_test_results_df
    .sort_values(
        "semantic_score",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    semantic_test_results_df[
        [
            "name",
            "semantic_score",
            "retrieval_text"
        ]
    ]
    .head(20)
)

,name,semantic_score,retrieval_text
0,1inch Limited,0.691172,Company: 1inch Limited\nDescription: 1inch Lim...
1,68 Labs,0.652177,Company: 68 Labs\nDescription: Creative tech a...
2,360AI,0.647315,Company: 360AI\nDescription: An early stage st...
3,1team.ai,0.644913,Company: 1team.ai\nDescription: 1team.ai is an...
4,41DEVS,0.640783,Company: 41DEVS\nDescription: Software as a Se...
5,1UNi,0.606862,Company: 1UNi\nDescription: 1UNi is a SaaS-ena...
6,1MillionResume,0.602808,"Company: 1MillionResume\nDescription: Saas, AI..."
7,12 X ONE,0.600521,Company: 12 X ONE\nDescription: One ecosystem ...
8,1st Hope Corps,0.594769,"Company: 1st Hope Corps\nDescription: SaaS, Mo..."
9,28Apps,0.592397,Company: 28Apps\nDescription: 28Apps specializ...


In [56]:
# CELL 46 - EMBED ALL 115,798 CANDIDATES

import time
import numpy as np


FULL_EMBEDDING_BATCH_SIZE = 64


full_candidate_texts = (
    software_it_candidates_df[
        "retrieval_text"
    ]
    .fillna("")
    .astype(str)
    .tolist()
)


print(
    "Candidates to embed:",
    len(full_candidate_texts)
)


start_time = time.perf_counter()


full_candidate_embeddings = (
    embedding_model.encode(
        full_candidate_texts,

        batch_size=FULL_EMBEDDING_BATCH_SIZE,

        show_progress_bar=True,

        convert_to_numpy=True,

        normalize_embeddings=True,
    )
)


elapsed_seconds = (
    time.perf_counter()
    - start_time
)


print("\nFULL EMBEDDING COMPLETE")

print(
    "Embedding shape:",
    full_candidate_embeddings.shape
)

print(
    "Elapsed minutes:",
    round(
        elapsed_seconds / 60,
        2
    )
)

print(
    "Rows / second:",
    round(
        len(full_candidate_texts)
        / elapsed_seconds,
        2
    )
)

Candidates to embed: 115798


Batches: 100%|██████████| 1810/1810 [50:54<00:00,  1.69s/it] 



FULL EMBEDDING COMPLETE
Embedding shape: (115798, 384)
Elapsed minutes: 51.03
Rows / second: 37.82


In [57]:
# CELL 47 - SAVE FULL CANDIDATE EMBEDDINGS

CANDIDATE_EMBEDDINGS_FILE = Path(
    "data/company_dataset/retrieval/software_it_candidate_embeddings.npy"
)


np.save(
    CANDIDATE_EMBEDDINGS_FILE,
    full_candidate_embeddings
)


print(
    "Embeddings saved:",
    CANDIDATE_EMBEDDINGS_FILE
)

print(
    "Shape:",
    full_candidate_embeddings.shape
)

print(
    "Dtype:",
    full_candidate_embeddings.dtype
)


print(
    "Size MB:",
    round(
        CANDIDATE_EMBEDDINGS_FILE.stat().st_size
        / 1024
        / 1024,
        2
    )
)

Embeddings saved: data\company_dataset\software_it_candidate_embeddings.npy
Shape: (115798, 384)
Dtype: float32
Size MB: 169.63


In [58]:
# CELL 48 - SAVE CANDIDATE METADATA

CANDIDATE_METADATA_FILE = Path(
    "data/company_dataset/retrieval/software_it_candidate_metadata.csv"
)


candidate_metadata_columns = [
    "id",
    "name",
    "short_description",
    "categories",
    "locations",
    "operating_status",
    "company_type",
    "ipo_status",
    "num_employees_enum",
    "employee_min",
    "employee_max",
    "founded_year",
    "company_age_years",
    "funding_total",
    "num_funding_rounds",
    "last_funding_type",
    "retrieval_text",
]


candidate_metadata_df = (
    software_it_candidates_df[
        candidate_metadata_columns
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


candidate_metadata_df.to_csv(
    CANDIDATE_METADATA_FILE,
    index=False
)


print(
    "Metadata saved:",
    CANDIDATE_METADATA_FILE
)

print(
    "Metadata rows:",
    len(candidate_metadata_df)
)

print(
    "Embedding rows:",
    len(full_candidate_embeddings)
)


assert (
    len(candidate_metadata_df)
    ==
    len(full_candidate_embeddings)
)


print(
    "Metadata / embedding alignment: OK"
)

Metadata saved: data\company_dataset\software_it_candidate_metadata.csv
Metadata rows: 115798
Embedding rows: 115798
Metadata / embedding alignment: OK


In [59]:
# CELL 49 - TEST RELOAD SAVED EMBEDDINGS AND METADATA

loaded_candidate_embeddings = np.load(
    CANDIDATE_EMBEDDINGS_FILE
)


loaded_candidate_metadata_df = pd.read_csv(
    CANDIDATE_METADATA_FILE,
    low_memory=False
)


print(
    "Loaded embeddings:",
    loaded_candidate_embeddings.shape
)

print(
    "Loaded metadata rows:",
    len(
        loaded_candidate_metadata_df
    )
)


assert (
    loaded_candidate_embeddings.shape[0]
    ==
    len(
        loaded_candidate_metadata_df
    )
)


print(
    "Reload alignment: OK"
)

Loaded embeddings: (115798, 384)
Loaded metadata rows: 115798
Reload alignment: OK


In [60]:
# CELL 50 - SEMANTIC SEARCH ACROSS ALL SAVED CANDIDATES

test_query = """
A small software startup building an AI-powered SaaS
platform for businesses. The company is early-stage,
has a small team, and provides software and information
technology services.
""".strip()


query_embedding = (
    embedding_model.encode(
        [test_query],

        convert_to_numpy=True,

        normalize_embeddings=True,
    )[0]
)


similarity_scores = (
    loaded_candidate_embeddings
    @ query_embedding
)


TOP_K = 20


top_indices = np.argpartition(
    similarity_scores,
    -TOP_K
)[
    -TOP_K:
]


top_indices = (
    top_indices[
        np.argsort(
            similarity_scores[
                top_indices
            ]
        )[::-1]
    ]
)


semantic_results_df = (
    loaded_candidate_metadata_df
    .iloc[
        top_indices
    ]
    .copy()
)


semantic_results_df[
    "semantic_score"
] = (
    similarity_scores[
        top_indices
    ]
)


semantic_results_df = (
    semantic_results_df
    .reset_index(
        drop=True
    )
)


display(
    semantic_results_df[
        [
            "name",
            "semantic_score",
            "short_description",
            "categories",
            "locations",
            "num_employees_enum",
            "founded_year",
        ]
    ]
)

,name,semantic_score,short_description,categories,locations,num_employees_enum,founded_year
0,SaaS Maker,0.772340,SaaS Maker offers a low-code platform for quic...,"Artificial Intelligence (AI), Information Tech...",Southern US,c_00001_00010,2018.0
1,Raiz,0.743156,A Saas platform that uses AI to match early st...,"Artificial Intelligence (AI), Machine Learning...","Dallas/Fort Worth Metroplex, Southern US",c_00001_00010,2021.0
2,SimpleAI,0.733930,SaaS platform leverages the power of artificia...,Software,"Asia-Pacific (APAC), Association of Southeast ...",c_00011_00050,2023.0
3,AI-PREDATOR,0.729971,SaaS - A Pro active AI that identifies and tar...,"Artificial Intelligence (AI), SaaS, Software","Europe, Middle East, and Africa (EMEA), Middle...",c_00001_00010,2019.0
4,StartingPoint,0.729774,SaaS workflow management software for service-...,"Customer Service, Management Information Syste...","Greater Los Angeles Area, West Coast, Western US",c_00001_00010,2020.0
5,Parsed,0.722866,SaaS collaborative workspace to create AI-auto...,"Artificial Intelligence (AI), Software","Great Lakes, Midwestern US",c_00001_00010,2022.0
6,SaaS Industries,0.721755,SaaS Industries provides an accelerator proces...,"SaaS, Software, Venture Capital","Greater Phoenix Area, Western US",c_00001_00010,2016.0
7,Gradient Labs,0.721053,We build AI SaaS tools that make people's live...,Software,NaN,c_00001_00010,2022.0
8,Personifwy,0.717020,"An AI enabled SAAS platform,that uses Advanced...","Analytics, Artificial Intelligence (AI), Emplo...",Asia-Pacific (APAC),c_00001_00010,2019.0
9,IllumiDesk,0.709027,SaaS product that allows teaching professional...,"E-Learning, Education, Information Technology,...","Greater Atlanta Area, East Coast, Southern US",c_00001_00010,2015.0


In [61]:
# CELL 51 - DEFINE REUSABLE SEMANTIC COMPANY SEARCH

def search_similar_companies(
    query,
    *,
    top_k=20,
    embeddings=None,
    metadata_df=None,
    model=None,
):
    """
    Semantic search across precomputed company embeddings.
    """

    if embeddings is None:
        embeddings = loaded_candidate_embeddings

    if metadata_df is None:
        metadata_df = loaded_candidate_metadata_df

    if model is None:
        model = embedding_model


    # ----------------------------------
    # Embed query only
    # ----------------------------------

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0]


    # ----------------------------------
    # Cosine similarity
    # embeddings already normalized
    # ----------------------------------

    similarity_scores = (
        embeddings
        @ query_embedding
    )


    top_k = min(
        top_k,
        len(similarity_scores)
    )


    # Efficient top-k selection
    top_indices = np.argpartition(
        similarity_scores,
        -top_k
    )[
        -top_k:
    ]


    top_indices = top_indices[
        np.argsort(
            similarity_scores[
                top_indices
            ]
        )[::-1]
    ]


    # ----------------------------------
    # Build results
    # ----------------------------------

    results = (
        metadata_df
        .iloc[
            top_indices
        ]
        .copy()
    )


    results[
        "semantic_score"
    ] = (
        similarity_scores[
            top_indices
        ]
    )


    results = (
        results
        .reset_index(drop=True)
    )


    return results

In [62]:
# CELL 52 - TEST REUSABLE SEARCH FUNCTION

test_query = """
A small software startup building an AI-powered SaaS
platform for businesses. The company is early-stage,
has a small team, and provides software and information
technology services.
""".strip()


search_results_df = (
    search_similar_companies(
        test_query,
        top_k=20
    )
)


display(
    search_results_df[
        [
            "name",
            "semantic_score",
            "short_description",
            "categories",
            "locations",
            "num_employees_enum",
            "founded_year",
        ]
    ]
)

,name,semantic_score,short_description,categories,locations,num_employees_enum,founded_year
0,SaaS Maker,0.772340,SaaS Maker offers a low-code platform for quic...,"Artificial Intelligence (AI), Information Tech...",Southern US,c_00001_00010,2018.0
1,Raiz,0.743156,A Saas platform that uses AI to match early st...,"Artificial Intelligence (AI), Machine Learning...","Dallas/Fort Worth Metroplex, Southern US",c_00001_00010,2021.0
2,SimpleAI,0.733930,SaaS platform leverages the power of artificia...,Software,"Asia-Pacific (APAC), Association of Southeast ...",c_00011_00050,2023.0
3,AI-PREDATOR,0.729971,SaaS - A Pro active AI that identifies and tar...,"Artificial Intelligence (AI), SaaS, Software","Europe, Middle East, and Africa (EMEA), Middle...",c_00001_00010,2019.0
4,StartingPoint,0.729774,SaaS workflow management software for service-...,"Customer Service, Management Information Syste...","Greater Los Angeles Area, West Coast, Western US",c_00001_00010,2020.0
5,Parsed,0.722866,SaaS collaborative workspace to create AI-auto...,"Artificial Intelligence (AI), Software","Great Lakes, Midwestern US",c_00001_00010,2022.0
6,SaaS Industries,0.721755,SaaS Industries provides an accelerator proces...,"SaaS, Software, Venture Capital","Greater Phoenix Area, Western US",c_00001_00010,2016.0
7,Gradient Labs,0.721053,We build AI SaaS tools that make people's live...,Software,NaN,c_00001_00010,2022.0
8,Personifwy,0.717020,"An AI enabled SAAS platform,that uses Advanced...","Analytics, Artificial Intelligence (AI), Emplo...",Asia-Pacific (APAC),c_00001_00010,2019.0
9,IllumiDesk,0.709027,SaaS product that allows teaching professional...,"E-Learning, Education, Information Technology,...","Greater Atlanta Area, East Coast, Southern US",c_00001_00010,2015.0


In [63]:
# CELL 53 - BENCHMARK SEMANTIC SEARCH SPEED

import time


benchmark_query = """
AI software company providing SaaS tools
for small and medium businesses.
""".strip()


start_time = time.perf_counter()


benchmark_results_df = (
    search_similar_companies(
        benchmark_query,
        top_k=20
    )
)


elapsed_ms = (
    time.perf_counter()
    - start_time
) * 1000


print(
    "Search time:",
    round(
        elapsed_ms,
        2
    ),
    "ms"
)


print(
    "Results:",
    len(
        benchmark_results_df
    )
)


display(
    benchmark_results_df[
        [
            "name",
            "semantic_score",
            "short_description",
        ]
    ]
    .head(10)
)

Search time: 36.41 ms
Results: 20


,name,semantic_score,short_description
0,SaaS Maker,0.771042,SaaS Maker offers a low-code platform for quic...
1,Businessflow AI,0.759995,AI powered SaaS for the next generation recrui...
2,Snapvision.Tech,0.746031,AI based SaaS Product Development
3,AI-PREDATOR,0.744647,SaaS - A Pro active AI that identifies and tar...
4,Parsed,0.744270,SaaS collaborative workspace to create AI-auto...
5,SimpleAI,0.743154,SaaS platform leverages the power of artificia...
6,Digilytics AI,0.742381,"Revolutionizing Mortgage Origination, Leveragi..."
7,Integrate.ai,0.741892,Integrate.ai is a SaaS startup focused on enab...
8,Fastagger,0.741636,SaaS ML/AI company providing Edge AI software ...
9,Acuity.AI,0.740008,Acuity.AI is a SaaS that uses AI technology to...


In [76]:
# CELL 54 - FORMAT COMPANY SEARCH RESULTS

def format_company_search_results(
    results_df,
    *,
    top_k=10,
):
    """
    Convert semantic search dataframe into
    backend-friendly dictionaries.
    """

    formatted_results = []

    limited_df = (
        results_df
        .head(top_k)
        .copy()
    )


    for rank, (_, row) in enumerate(
        limited_df.iterrows(),
        start=1
    ):

        formatted_results.append(
            {
                "rank": rank,

                "id": (
                    None
                    if pd.isna(row["id"])
                    else str(row["id"])
                ),

                "name": (
                    None
                    if pd.isna(row["name"])
                    else str(row["name"])
                ),

                "semantic_score": round(
                    float(
                        row["semantic_score"]
                    ),
                    6
                ),

                "short_description": (
                    None
                    if pd.isna(
                        row["short_description"]
                    )
                    else str(
                        row["short_description"]
                    )
                ),

                "categories": (
                    None
                    if pd.isna(
                        row["categories"]
                    )
                    else str(
                        row["categories"]
                    )
                ),

                "locations": (
                    None
                    if pd.isna(
                        row["locations"]
                    )
                    else str(
                        row["locations"]
                    )
                ),

                "employee_range": (
                    None
                    if pd.isna(
                        row["num_employees_enum"]
                    )
                    else employee_range_text_mapping.get(
                        str(
                            row["num_employees_enum"]
                        ),
                        str(
                            row["num_employees_enum"]
                        )
                    )
                ),

                "founded_year": (
                    None
                    if pd.isna(
                        row["founded_year"]
                    )
                    else int(
                        float(
                            row["founded_year"]
                        )
                    )
                ),
            }
        )


    return formatted_results

In [77]:
# CELL 55 - DEFINE COMPANY ANALOGUE RETRIEVAL PIPELINE

def retrieve_company_analogues(
    query,
    *,
    top_k=10,
):
    """
    Retrieve semantically similar company analogues
    from the precomputed candidate universe.
    """

    search_results = (
        search_similar_companies(
            query,
            top_k=top_k
        )
    )


    formatted_results = (
        format_company_search_results(
            search_results,
            top_k=top_k
        )
    )


    return {
        "query": query,
        "candidate_universe_size":
            len(
                loaded_candidate_metadata_df
            ),

        "result_count":
            len(
                formatted_results
            ),

        "results":
            formatted_results,
    }

In [78]:
# CELL 56 - TEST FINAL COMPANY ANALOGUE RETRIEVAL

decision_query = """
A small early-stage AI SaaS company building
software tools for businesses to automate
operational workflows.
""".strip()


company_analogue_output = (
    retrieve_company_analogues(
        decision_query,
        top_k=10
    )
)


print(
    "Candidate universe:",
    company_analogue_output[
        "candidate_universe_size"
    ]
)

print(
    "Results:",
    company_analogue_output[
        "result_count"
    ]
)


for result in company_analogue_output[
    "results"
]:

    print("\n" + "=" * 80)

    print(
        f"#{result['rank']} "
        f"{result['name']}"
    )

    print(
        "Score:",
        result[
            "semantic_score"
        ]
    )

    print(
        "Description:",
        result[
            "short_description"
        ]
    )

    print(
        "Categories:",
        result[
            "categories"
        ]
    )

    print(
        "Location:",
        result[
            "locations"
        ]
    )

Candidate universe: 115798
Results: 10

#1 Aiver.ai
Score: 0.733088
Description: Unlock The Power Of Automation. Automate workflows with AIVER using the power of AI and ML to make the workflows smarter over time.
Categories: Accounting, Artificial Intelligence (AI), Business Intelligence, Compliance, Data Visualization, Fraud Detection, Information Technology, Machine Learning, Predictive Analytics, Product Management
Location: San Francisco Bay Area, Silicon Valley, West Coast, Western US

#2 Work Simplr
Score: 0.708981
Description: AI driven platform to place and manage vetted, qualified students for paid short-term, remote entry level work .
Categories: Business Process Automation (BPA), EdTech, Freelance, Human Resources, Outsourcing, Professional Services, Recruiting, Social Entrepreneurship, Software, Staffing Agency
Location: Greater Denver Area, Western US

#3 Handy.ai
Score: 0.705028
Description: Handy.ai combines process automation with AI agents to speed up business operatio

In [79]:
# CELL 57 - RETRIEVE COMPANY ANALOGUES FOR MULTIPLE ASSUMPTIONS

def retrieve_analogues_for_assumptions(
    assumptions,
    *,
    top_k_per_assumption=5,
):
    """
    Run company analogue retrieval for multiple assumptions.

    Parameters
    ----------
    assumptions : list[str] or list[dict]
        Supports:
        [
            "Assumption text",
            ...
        ]

        or:

        [
            {
                "id": "...",
                "assumption": "..."
            }
        ]

    top_k_per_assumption : int
        Number of analogue companies returned
        for each assumption.
    """

    outputs = []


    for index, item in enumerate(
        assumptions,
        start=1
    ):

        # --------------------------------
        # Normalize assumption input
        # --------------------------------

        if isinstance(item, dict):

            assumption_id = (
                item.get("id")
                or f"assumption_{index}"
            )

            assumption_text = (
                item.get("assumption")
                or item.get("text")
                or item.get("description")
                or ""
            )

        else:

            assumption_id = (
                f"assumption_{index}"
            )

            assumption_text = str(
                item
            )


        assumption_text = (
            assumption_text
            .strip()
        )


        if not assumption_text:
            continue


        # --------------------------------
        # Retrieve analogues
        # --------------------------------

        retrieval_output = (
            retrieve_company_analogues(
                assumption_text,
                top_k=top_k_per_assumption
            )
        )


        outputs.append(
            {
                "assumption_id":
                    assumption_id,

                "assumption":
                    assumption_text,

                "candidate_universe_size":
                    retrieval_output[
                        "candidate_universe_size"
                    ],

                "analogue_count":
                    retrieval_output[
                        "result_count"
                    ],

                "analogues":
                    retrieval_output[
                        "results"
                    ],
            }
        )


    return outputs

In [80]:
# CELL 58 - TEST MULTI-ASSUMPTION RETRIEVAL

test_assumptions = [
    {
        "id": "A1",
        "assumption": (
            "Small businesses are willing to pay "
            "for AI-powered workflow automation software."
        )
    },

    {
        "id": "A2",
        "assumption": (
            "A small software team can build and operate "
            "a scalable SaaS platform for business customers."
        )
    },

    {
        "id": "A3",
        "assumption": (
            "Businesses need software tools that automate "
            "repetitive operational processes."
        )
    },
]


multi_assumption_results = (
    retrieve_analogues_for_assumptions(
        test_assumptions,
        top_k_per_assumption=5
    )
)


print(
    "Assumptions processed:",
    len(
        multi_assumption_results
    )
)


for assumption_result in multi_assumption_results:

    print(
        "\n" + "=" * 100
    )

    print(
        "Assumption ID:",
        assumption_result[
            "assumption_id"
        ]
    )

    print(
        "Assumption:",
        assumption_result[
            "assumption"
        ]
    )

    print(
        "Analogue count:",
        assumption_result[
            "analogue_count"
        ]
    )


    for analogue in assumption_result[
        "analogues"
    ]:

        print(
            f"\n  #{analogue['rank']} "
            f"{analogue['name']}"
        )

        print(
            "  Score:",
            analogue[
                "semantic_score"
            ]
        )

        print(
            "  Description:",
            analogue[
                "short_description"
            ]
        )

Assumptions processed: 3

Assumption ID: A1
Assumption: Small businesses are willing to pay for AI-powered workflow automation software.
Analogue count: 5

  #1 Aiver.ai
  Score: 0.667223
  Description: Unlock The Power Of Automation. Automate workflows with AIVER using the power of AI and ML to make the workflows smarter over time.

  #2 MiniMax
  Score: 0.656396
  Description: MiniMax develops AI technology for social connections and interaction, converting text into visual, audio, and text to text modes.

  #3 Latent AI
  Score: 0.65275
  Description: Latent AI accelerates AI implementation and workflows for the enterprise cost-effectively anywhere on the edge continuum with Adaptive AI.

  #4 Workflows
  Score: 0.651271
  Description: Automation using bots.

  #5 Facere.AI
  Score: 0.645491
  Description: Artificial Intelligent Powered Workflow Automation For Healthcare Industries

Assumption ID: A2
Assumption: A small software team can build and operate a scalable SaaS platform fo

In [81]:
# CELL 59 - BUILD BACKEND-FRIENDLY COMPANY ANALOGUE OUTPUT

def build_company_analogue_response(
    assumptions,
    *,
    top_k_per_assumption=5,
):

    assumption_results = (
        retrieve_analogues_for_assumptions(
            assumptions,
            top_k_per_assumption=
                top_k_per_assumption
        )
    )


    total_analogues = sum(
        item[
            "analogue_count"
        ]
        for item
        in assumption_results
    )


    return {
        "assumption_count":
            len(
                assumption_results
            ),

        "total_analogue_results":
            total_analogues,

        "candidate_universe_size":
            len(
                loaded_candidate_metadata_df
            ),

        "assumptions":
            assumption_results,
    }


company_analogue_response = (
    build_company_analogue_response(
        test_assumptions,
        top_k_per_assumption=5
    )
)


print(
    "Assumption count:",
    company_analogue_response[
        "assumption_count"
    ]
)

print(
    "Total analogue results:",
    company_analogue_response[
        "total_analogue_results"
    ]
)

print(
    "Candidate universe:",
    company_analogue_response[
        "candidate_universe_size"
    ]
)

Assumption count: 3
Total analogue results: 15
Candidate universe: 115798


In [82]:
# CELL 60 - NORMALIZE ASSUMPTION ANALYZER OUTPUT

def normalize_analyzer_assumptions(
    analyzer_output
):
    """
    Convert different assumption-analyzer output shapes
    into a consistent format:

    [
        {
            "id": "...",
            "assumption": "...",
            "retrieval_query": "..."
        }
    ]
    """

    normalized = []


    # -----------------------------------
    # Find assumption list
    # -----------------------------------

    if isinstance(
        analyzer_output,
        dict
    ):

        if isinstance(
            analyzer_output.get(
                "assumptions"
            ),
            list
        ):

            raw_assumptions = (
                analyzer_output[
                    "assumptions"
                ]
            )

        elif isinstance(
            analyzer_output.get(
                "results"
            ),
            list
        ):

            raw_assumptions = (
                analyzer_output[
                    "results"
                ]
            )

        else:

            raw_assumptions = [
                analyzer_output
            ]


    elif isinstance(
        analyzer_output,
        list
    ):

        raw_assumptions = (
            analyzer_output
        )


    else:

        raw_assumptions = [
            analyzer_output
        ]


    # -----------------------------------
    # Normalize each assumption
    # -----------------------------------

    for index, item in enumerate(
        raw_assumptions,
        start=1
    ):

        if isinstance(
            item,
            dict
        ):

            assumption_id = (
                item.get("id")
                or item.get("assumption_id")
                or item.get("key")
                or f"A{index}"
            )


            assumption_text = (
                item.get("assumption")
                or item.get("assumption_text")
                or item.get("text")
                or item.get("description")
                or item.get("statement")
                or ""
            )


            retrieval_query = (
                item.get("retrieval_query")
                or item.get("semantic_query")
                or item.get("search_query")
                or item.get("query")
                or assumption_text
            )


        else:

            assumption_id = (
                f"A{index}"
            )

            assumption_text = (
                str(item)
            )

            retrieval_query = (
                assumption_text
            )


        assumption_text = (
            str(assumption_text)
            .strip()
        )


        retrieval_query = (
            str(retrieval_query)
            .strip()
        )


        if not assumption_text:
            continue


        if not retrieval_query:
            retrieval_query = (
                assumption_text
            )


        normalized.append(
            {
                "id":
                    str(
                        assumption_id
                    ),

                "assumption":
                    assumption_text,

                "retrieval_query":
                    retrieval_query,
            }
        )


    return normalized

In [83]:
# CELL 61 - RETRIEVE ANALOGUES FROM ANALYZER OUTPUT

def retrieve_company_analogues_from_analyzer(
    analyzer_output,
    *,
    top_k_per_assumption=5,
):
    """
    Normalize analyzer output, then retrieve
    company analogues for each assumption.
    """

    normalized_assumptions = (
        normalize_analyzer_assumptions(
            analyzer_output
        )
    )


    assumption_outputs = []


    for item in normalized_assumptions:

        retrieval_output = (
            retrieve_company_analogues(
                item[
                    "retrieval_query"
                ],
                top_k=
                    top_k_per_assumption
            )
        )


        assumption_outputs.append(
            {
                "assumption_id":
                    item["id"],

                "assumption":
                    item[
                        "assumption"
                    ],

                "retrieval_query":
                    item[
                        "retrieval_query"
                    ],

                "candidate_universe_size":
                    retrieval_output[
                        "candidate_universe_size"
                    ],

                "analogue_count":
                    retrieval_output[
                        "result_count"
                    ],

                "analogues":
                    retrieval_output[
                        "results"
                    ],
            }
        )


    return {
        "assumption_count":
            len(
                assumption_outputs
            ),

        "candidate_universe_size":
            len(
                loaded_candidate_metadata_df
            ),

        "assumptions":
            assumption_outputs,
    }

In [84]:
# CELL 62 - TEST ANALYZER → COMPANY ANALOGUE PIPELINE

mock_analyzer_output = {
    "assumptions": [
        {
            "id": "A1",

            "assumption": (
                "Small businesses are willing "
                "to pay for AI-powered workflow "
                "automation software."
            ),

            "retrieval_query": (
                "Small AI SaaS companies providing "
                "workflow automation software to "
                "business customers."
            ),
        },

        {
            "id": "A2",

            "assumption": (
                "A small engineering team can "
                "operate a scalable SaaS platform."
            ),

            "retrieval_query": (
                "Small software SaaS companies "
                "with small teams building scalable "
                "business software platforms."
            ),
        },

        {
            "id": "A3",

            "assumption": (
                "Businesses have meaningful demand "
                "for software that automates "
                "repetitive operational tasks."
            ),

            "retrieval_query": (
                "Business software companies "
                "providing operational workflow "
                "automation tools."
            ),
        },
    ]
}


analyzer_company_results = (
    retrieve_company_analogues_from_analyzer(
        mock_analyzer_output,
        top_k_per_assumption=5
    )
)


print(
    "Assumptions:",
    analyzer_company_results[
        "assumption_count"
    ]
)

print(
    "Candidate universe:",
    analyzer_company_results[
        "candidate_universe_size"
    ]
)


for item in analyzer_company_results[
    "assumptions"
]:

    print(
        "\n" + "=" * 100
    )

    print(
        "Assumption ID:",
        item[
            "assumption_id"
        ]
    )

    print(
        "Assumption:",
        item[
            "assumption"
        ]
    )

    print(
        "Retrieval query:",
        item[
            "retrieval_query"
        ]
    )


    for analogue in item[
        "analogues"
    ]:

        print(
            f"\n  #{analogue['rank']} "
            f"{analogue['name']}"
        )

        print(
            "  Score:",
            analogue[
                "semantic_score"
            ]
        )

        print(
            "  Description:",
            analogue[
                "short_description"
            ]
        )

Assumptions: 3
Candidate universe: 115798

Assumption ID: A1
Assumption: Small businesses are willing to pay for AI-powered workflow automation software.
Retrieval query: Small AI SaaS companies providing workflow automation software to business customers.

  #1 Businessflow AI
  Score: 0.732168
  Description: AI powered SaaS for the next generation recruitment.

  #2 Aiver.ai
  Score: 0.730408
  Description: Unlock The Power Of Automation. Automate workflows with AIVER using the power of AI and ML to make the workflows smarter over time.

  #3 Handy.ai
  Score: 0.723168
  Description: Handy.ai combines process automation with AI agents to speed up business operations for small and medium sized businesses.

  #4 Work Simplr
  Score: 0.716551
  Description: AI driven platform to place and manage vetted, qualified students for paid short-term, remote entry level work .

  #5 Workorder AI
  Score: 0.714209
  Description: Workorder AI is an automated quantity takeoff software for construct

In [85]:
# CELL 63 - FINAL COMPANY ANALOGUE INTEGRATION WRAPPER

def build_company_analogue_evidence(
    analyzer_output,
    *,
    top_k_per_assumption=5,
):
    """
    Final integration wrapper for company analogue evidence.

    Input:
        Raw assumption analyzer output.

    Output:
        Backend-friendly company analogue evidence
        grouped by assumption.
    """

    retrieval_result = (
        retrieve_company_analogues_from_analyzer(
            analyzer_output,
            top_k_per_assumption=
                top_k_per_assumption
        )
    )


    total_analogues = sum(
        item["analogue_count"]
        for item
        in retrieval_result["assumptions"]
    )


    return {
        "source": "large_company_dataset",

        "candidate_universe_size":
            retrieval_result[
                "candidate_universe_size"
            ],

        "assumption_count":
            retrieval_result[
                "assumption_count"
            ],

        "total_analogue_results":
            total_analogues,

        "assumptions":
            retrieval_result[
                "assumptions"
            ],
    }

In [86]:
# CELL 64 - TEST FINAL COMPANY ANALOGUE EVIDENCE PAYLOAD

company_analogue_evidence = (
    build_company_analogue_evidence(
        mock_analyzer_output,
        top_k_per_assumption=5
    )
)


print(
    "Source:",
    company_analogue_evidence[
        "source"
    ]
)

print(
    "Candidate universe:",
    company_analogue_evidence[
        "candidate_universe_size"
    ]
)

print(
    "Assumptions:",
    company_analogue_evidence[
        "assumption_count"
    ]
)

print(
    "Total analogues:",
    company_analogue_evidence[
        "total_analogue_results"
    ]
)

Source: large_company_dataset
Candidate universe: 115798
Assumptions: 3
Total analogues: 15


In [87]:
# CELL 65 - INSPECT FINAL JSON-LIKE OUTPUT

import json


print(
    json.dumps(
        company_analogue_evidence,
        indent=2,
        ensure_ascii=False
    )[:15000]
)

{
  "source": "large_company_dataset",
  "candidate_universe_size": 115798,
  "assumption_count": 3,
  "total_analogue_results": 15,
  "assumptions": [
    {
      "assumption_id": "A1",
      "assumption": "Small businesses are willing to pay for AI-powered workflow automation software.",
      "retrieval_query": "Small AI SaaS companies providing workflow automation software to business customers.",
      "candidate_universe_size": 115798,
      "analogue_count": 5,
      "analogues": [
        {
          "rank": 1,
          "id": "a4e20fc6-334e-44a5-8cae-0d7c5908016e",
          "name": "Businessflow AI",
          "semantic_score": 0.732168,
          "short_description": "AI powered SaaS for the next generation recruitment.",
          "categories": "Artificial Intelligence (AI), Business Information Systems, Business Process Automation (BPA), Enterprise Software, Human Resources, Recruiting, Software",
          "locations": null,
          "employee_range": "1-10 employees",
 